# CTIBench Contamination Audit - Stage 1: Matched Sampling, Baselines, Scoring Pilot

Stage 0 passed GREEN: 34,959 clean post-cutoff CVEs, 100% coverage of the benchmark's
104 CWE classes. This notebook does four things:

1. **Exact-histogram matching** - build a post-cutoff arm whose CWE distribution is
   identical to the benchmark's, class for class. Removes the distribution confound
   by construction rather than correcting for it.
2. **Non-LLM baselines** - majority class and TF-IDF. Run these BEFORE spending on any
   API. If TF-IDF lands near published LLM numbers, that is itself the paper.
3. **Provider-agnostic scoring harness** - any OpenAI-compatible endpoint, bounded
   concurrency, resumable checkpoints, tools explicitly disabled.
4. **Analysis** - two-proportion test with bootstrap CIs, plus free CVSS component-wise
   decomposition.

**Runtime: CPU.** Everything here is network-bound or light sklearn.

**Prerequisite:** Stage 0 must have written the raw snapshot to
`/content/drive/MyDrive/ctibench-audit/raw/`.

## 1. Setup and config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, csv, re, time, random, hashlib, threading, subprocess
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

PROJECT = Path('/content/drive/MyDrive/ctibench-audit')
RAW     = PROJECT / 'raw'
DERIVED = PROJECT / 'derived'
RESULTS = PROJECT / 'results'
for p in (DERIVED, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

SEED = 20261005
random.seed(SEED)

# ---- PILOT KNOBS -------------------------------------------------------
PILOT_N   = 999        # per arm. Raise to 999 for the full run.
                       # NOTE: rare classes get a floor of 1, so at N=300 the tail is
                       # over-weighted vs the benchmark's true histogram (you get ~332
                       # items). Both arms share the same target so the COMPARISON is
                       # clean, but pilot accuracies are NOT comparable to CTIBench's
                       # published figures. Only N=999 reproduces the real distribution.
TASKS     = ['rcm']    # add 'vsp' once rcm looks right; vsp prompts are ~4x longer
MAX_WORKERS = 8        # concurrent requests; lower if you hit 429s
# ------------------------------------------------------------------------

# CUTOFF BOUNDARY: post-cutoff arm uses CVEs published strictly after this date.
# Must be >= the LATEST cutoff among models in the temporal arm.
# Verify every date on the provider's own model card and record the URL.
CUTOFF_BOUNDARY = '2026-06-01'

print('project:', PROJECT)
print('boundary:', CUTOFF_BOUNDARY, '| pilot N:', PILOT_N, '| tasks:', TASKS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project: /content/drive/MyDrive/ctibench-audit
boundary: 2026-06-01 | pilot N: 999 | tasks: ['rcm']


## 2. Load benchmark + frozen snapshot

In [ ]:
if not Path('/content/cti-bench').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xashru/cti-bench.git',
                    '/content/cti-bench'], check=True)
DATA = Path('/content/cti-bench/data')

def load_tsv(name):
    with open(DATA / name, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f, delimiter='\t'))
    for r in rows:
        r['GT'] = r['GT'].strip()
    return rows

rcm_bench = load_tsv('cti-rcm.tsv')
vsp_bench = load_tsv('cti-vsp.tsv')

BENCH_HIST = Counter(r['GT'] for r in rcm_bench)
print('benchmark items:', len(rcm_bench), '| distinct CWEs:', len(BENCH_HIST))

# Match ONLY the Stage 0 forward snapshot. Stage 1b writes nvd_old_*.jsonl into the
# same folder, and a bare nvd_*.jsonl glob would sort 'old' last and silently load the
# 2023-24 file instead.
snaps = sorted(p for p in RAW.glob('nvd_2*.jsonl') if not p.name.startswith('nvd_old'))
assert snaps, ('No Stage 0 forward snapshot (nvd_2*.jsonl) in raw/. Run Stage 0 first.')
SNAP = snaps[-1]
print('snapshot:', SNAP.name)
if len(snaps) > 1:
    print('  NOTE: multiple forward snapshots present; using the newest.')
    for p in snaps:
        print('   -', p.name)

benchmark items: 1000 | distinct CWEs: 104
snapshot: nvd_2026-01-01_2026-09-05_pulled2026-09-05.jsonl


In [ ]:
BAD_CWE = {'NVD-CWE-noinfo', 'NVD-CWE-Other', 'NVD-CWE-Unknown'}
BENCH_LABELS = set(BENCH_HIST)

def extract(v):
    c = v['cve']
    desc = next((d['value'] for d in c.get('descriptions', []) if d.get('lang') == 'en'), None)
    cwes = {d['value'] for w in c.get('weaknesses', [])
            for d in w.get('description', []) if d.get('value', '').startswith('CWE-')}
    m = (c.get('metrics') or {}).get('cvssMetricV31') or []
    vec = score = None
    if m:
        cd = m[0].get('cvssData', {})
        vec, score = cd.get('vectorString'), cd.get('baseScore')
    return {'cve': c.get('id'), 'published': c.get('published'),
            'status': c.get('vulnStatus'), 'description': desc,
            'cwes': sorted(cwes), 'cvss_vector': vec, 'cvss_score': score}

def survives(r):
    if not r['description'] or len(r['description']) < 40: return False
    if r['status'] in ('Awaiting Analysis', 'Received', 'Undergoing Analysis'): return False
    good = [c for c in r['cwes'] if c not in BAD_CWE and c in BENCH_LABELS]
    if len(good) != 1: return False
    if not r['cvss_vector'] or r['cvss_score'] is None: return False
    r['GT_cwe'] = good[0]
    return True

clean, seen = [], set()
with open(SNAP, encoding='utf-8') as f:
    for line in f:
        r = extract(json.loads(line))
        if r['cve'] and r['cve'] not in seen and survives(r):
            seen.add(r['cve']); clean.append(r)

# Apply the cutoff boundary. This is the filter that replaces re-pulling.
post = [r for r in clean if r['published'][:10] > CUTOFF_BOUNDARY]
post_pool_all = post
print(f'clean total          {len(clean)}')
print(f'published > {CUTOFF_BOUNDARY}  {len(post)}   <- post-cutoff pool')

# Guard: a snapshot that cannot supply post-cutoff items means the wrong file loaded.
assert len(post) >= 1000, (
    f'Only {len(post)} CVEs published after {CUTOFF_BOUNDARY} in {SNAP.name}. '
    'Wrong snapshot loaded, or the boundary is past the snapshot end date. '
    'Stage 0 should have produced a Jan-Sep 2026 file.')
spans = (min(r['published'][:10] for r in clean), max(r['published'][:10] for r in clean))
print(f'snapshot covers      {spans[0]} .. {spans[1]}')

clean total          34959
published > 2026-06-01  16916   <- post-cutoff pool
snapshot covers      2026-01-01 .. 2026-09-04


## 3. Exact-histogram matching

Draw a post-cutoff sample whose CWE histogram matches the benchmark's, class for class,
scaled to `PILOT_N`. Any class the pool cannot satisfy is reported explicitly - never
silently substituted.

In [ ]:
pool = defaultdict(list)
for r in post:
    pool[r['GT_cwe']].append(r)

bench_by_cwe = defaultdict(list)
for r in rcm_bench:
    bench_by_cwe[r['GT']].append(r)

# COMMON LABEL SPACE: a class must exist in BOTH arms or it is excluded from both.
# Classes absent from the post-cutoff pool are absent permanently at this boundary -
# raising N will not recover them. Excluding them makes histogram drift zero by
# construction; leaving them in makes the arms structurally different.
COMMON = {c for c in BENCH_HIST if pool.get(c)}
EXCLUDED = {c: BENCH_HIST[c] for c in BENCH_HIST if c not in COMMON}
excl_mass = sum(EXCLUDED.values()) / sum(BENCH_HIST.values())

print(f'benchmark classes      : {len(BENCH_HIST)}')
print(f'present post-cutoff    : {len(COMMON)}')
print(f'excluded (absent post) : {len(EXCLUDED)}  = {100*excl_mass:.2f}% of benchmark mass')
for c, n in sorted(EXCLUDED.items(), key=lambda x: -x[1]):
    print(f'   {c}: {n} benchmark item(s)')
print('REPORT THIS EXCLUSION IN THE PAPER.')
print()

# Renormalise over the common label space
common_total = sum(BENCH_HIST[c] for c in COMMON)
scale = PILOT_N / common_total
target = {c: max(1, round(BENCH_HIST[c] * scale)) for c in COMMON}

# Largest N this pool can satisfy exactly, so you know if 999 is reachable
feasible = min(len(pool[c]) / (BENCH_HIST[c] / common_total) for c in COMMON)
print(f'max exactly-satisfiable N at this boundary: {int(feasible)}')
if feasible < 999:
    print('  -> below 999. Either lower N, or move CUTOFF_BOUNDARY earlier if your '
          'model roster allows it.')

post_arm, bench_arm, shortfalls = [], [], []
for cwe, need in target.items():
    ap, ab = pool[cwe], bench_by_cwe[cwe]
    take = min(need, len(ap), len(ab))
    if take < need:
        shortfalls.append((cwe, need, len(ap), len(ab)))
    post_arm  += random.sample(ap, take)
    bench_arm += random.sample(ab, take)

print(f'post-cutoff arm : {len(post_arm)}')
print(f'benchmark arm   : {len(bench_arm)}')
print(f'classes short   : {len(shortfalls)}')
for cwe, need, hp_, hb_ in shortfalls[:10]:
    print(f'   {cwe}: needed {need}, post={hp_}, bench={hb_}')

# ---- Freeze the arms. Once drawn, they are reused verbatim, so later code
# ---- changes cannot silently redraw a different sample and corrupt checkpoints.
ARM_FILE = DERIVED / f'arms_N{PILOT_N}_{CUTOFF_BOUNDARY}.json'
if ARM_FILE.exists():
    saved = json.loads(ARM_FILE.read_text())
    pidx = {r['cve']: r for r in post_pool_all}
    bidx = {re.search(r'CVE-\d{4}-\d+', r['URL']).group(0): r for r in rcm_bench}
    post_arm  = [pidx[c] for c in saved['post']  if c in pidx]
    bench_arm = [bidx[c] for c in saved['bench'] if c in bidx]
    print(f'REUSED frozen arms from {ARM_FILE.name}')
else:
    ARM_FILE.write_text(json.dumps({
        'post':  [r['cve'] for r in post_arm],
        'bench': [re.search(r'CVE-\d{4}-\d+', r['URL']).group(0) for r in bench_arm],
        'seed': SEED, 'N': PILOT_N, 'boundary': CUTOFF_BOUNDARY}))
    print(f'FROZE arms to {ARM_FILE.name} - delete this file to redraw')

hp = Counter(r['GT_cwe'] for r in post_arm)
hb = Counter(r['GT'] for r in bench_arm)
drift = sum(abs(hp[c] - hb[c]) for c in set(hp) | set(hb))
print(f'histogram drift between arms: {drift} items - MUST be 0')
assert drift == 0, 'Arms are not distribution-matched; do not proceed.'

benchmark classes      : 104
present post-cutoff    : 102
excluded (absent post) : 2  = 0.30% of benchmark mass
   CWE-257: 2 benchmark item(s)
   CWE-300: 1 benchmark item(s)
REPORT THIS EXCLUSION IN THE PAPER.

max exactly-satisfiable N at this boundary: 1566
post-cutoff arm : 997
benchmark arm   : 997
classes short   : 0
FROZE arms to arms_N999_2026-06-01.json - delete this file to redraw
histogram drift between arms: 0 items - MUST be 0


## 4. Non-LLM baselines - run before spending anything

Two reference points every LLM number must be read against:

- **Majority class** - predict CWE-79 always.
- **TF-IDF + logistic regression** - trained on post-cutoff CVEs *not* in either arm,
  so it has no advantage from memorization. If this lands near published LLM accuracy,
  the benchmark is measuring surface lexical cues, which is a finding in itself.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

maj = BENCH_HIST.most_common(1)[0][0]
print(f'majority class = {maj}')
print(f'  benchmark arm : {100*sum(r["GT"]==maj for r in bench_arm)/len(bench_arm):.1f}%')
print(f'  post-cutoff   : {100*sum(r["GT_cwe"]==maj for r in post_arm)/len(post_arm):.1f}%')

used = {r['cve'] for r in post_arm}
train = [r for r in clean if r['cve'] not in used]
Xtr = [r['description'] for r in train]
ytr = [r['GT_cwe'] for r in train]
print(f'\nTF-IDF training pool: {len(train)} CVEs (disjoint from both arms)')

clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True),
    LogisticRegression(max_iter=2000, C=4.0, n_jobs=-1))
clf.fit(Xtr, ytr)

pb = clf.predict([r['Description'] for r in bench_arm])
pp = clf.predict([r['description'] for r in post_arm])
acc_b = sum(p == r['GT'] for p, r in zip(pb, bench_arm)) / len(bench_arm)
acc_p = sum(p == r['GT_cwe'] for p, r in zip(pp, post_arm)) / len(post_arm)
print(f'TF-IDF  benchmark arm : {100*acc_b:.1f}%')
print(f'TF-IDF  post-cutoff   : {100*acc_p:.1f}%')
print('\nNOTE: TF-IDF has no memorization channel. Any gap it shows between arms is '
      'intrinsic task difficulty and must be SUBTRACTED from the LLM gap.')

majority class = CWE-79
  benchmark arm : 23.0%
  post-cutoff   : 23.0%

TF-IDF training pool: 33962 CVEs (disjoint from both arms)
TF-IDF  benchmark arm : 65.4%
TF-IDF  post-cutoff   : 79.9%

NOTE: TF-IDF has no memorization channel. Any gap it shows between arms is intrinsic task difficulty and must be SUBTRACTED from the LLM gap.


## 5. Scoring harness

Provider-agnostic: anything exposing an OpenAI-compatible `/chat/completions`.
Works with OpenAI, OpenRouter, Groq, Together, Fireworks, and Google AI Studio's
compatibility endpoint.

Add keys in Colab Secrets, or leave blank to be prompted.

In [ ]:
def get_key(name):
    try:
        from google.colab import userdata
        v = (userdata.get(name) or '').strip()
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ').strip()

# Register only the providers you actually have. Cutoffs MUST come from the
# provider's own model card - paste the URL in `cutoff_src`.
PROVIDERS = {
    'openrouter': {'base': 'https://openrouter.ai/api/v1',
                   'key_name': 'OPENROUTER_API_KEY'},
    'openai':     {'base': 'https://api.openai.com/v1',
                   'key_name': 'OPENAI_API_KEY'},
    'xai':        {'base': 'https://api.x.ai/v1',
                   'key_name': 'XAI_API_KEY'},
    'gemini':     {'base': 'https://generativelanguage.googleapis.com/v1beta/openai',
                   'key_name': 'GEMINI_API_KEY'},
    'groq':       {'base': 'https://api.groq.com/openai/v1',
                   'key_name': 'GROQ_API_KEY'},
}

# Which providers you intend to use. Keys are prompted only for these.
ACTIVE = ['openai']

MODELS = [
    # {'id': 'pilot-a', 'provider': 'groq', 'model': '<exact model string>',
    #  'cutoff': 'YYYY-MM-DD', 'cutoff_src': 'https://...model-card'},
]

KEYS = {p: get_key(PROVIDERS[p]['key_name']) for p in ACTIVE}
print('keys loaded for:', ACTIVE)

OPENAI_API_KEY: ··········
keys loaded for: ['openai']


In [ ]:
import requests

RCM_PROMPT = ('Analyze the following CVE description and map it to the appropriate CWE. '
              'Provide a brief justification for your choice. Ensure the last line of your '
              'response contains only the CWE ID.  CVE Description: ')
VSP_HEAD = vsp_bench[0]['Prompt'].split(vsp_bench[0]['Description'])[0]

def build_prompt(task, desc):
    return (RCM_PROMPT + desc + ' ') if task == 'rcm' else (VSP_HEAD + desc + ' ')

_lock = threading.Lock()

# Newer models reject 'max_tokens' (want 'max_completion_tokens') and reject
# non-default 'temperature'. Learn the right shape per model on first 400 and cache it.
PARAM_STYLE = {}

def _body(m, prompt):
    style = PARAM_STYLE.get(m['id'], {'tokens': 'max_tokens', 'temperature': True})
    b = {'model': m['model'], 'messages': [{'role': 'user', 'content': prompt}],
         style['tokens']: 600}
    if style['temperature']:
        b['temperature'] = 0
    return b

def _adapt(m, err_text):
    # Returns True if the parameter style was changed and the call is worth retrying.
    style = PARAM_STYLE.get(m['id'], {'tokens': 'max_tokens', 'temperature': True})
    changed = False
    if 'max_completion_tokens' in err_text and style['tokens'] == 'max_tokens':
        style['tokens'] = 'max_completion_tokens'; changed = True
    if 'temperature' in err_text and style['temperature']:
        style['temperature'] = False; changed = True
    if changed:
        PARAM_STYLE[m['id']] = style
        print(f"   [adapt] {m['id']} -> tokens={style['tokens']} "
              f"temperature={'on' if style['temperature'] else 'off'}")
    return changed

def call(m, prompt, timeout=120, tries=4, verbose=False):
    url = PROVIDERS[m['provider']]['base'] + '/chat/completions'
    key = KEYS.get(m['provider'], '')
    if not key:
        return None, {'error': f"no API key loaded for provider '{m['provider']}'"}
    hdr = {'Authorization': 'Bearer ' + key, 'Content-Type': 'application/json'}
    last = 'no attempt made'
    for i in range(tries):
        try:
            r = requests.post(url, json=_body(m, prompt), headers=hdr, timeout=timeout)
            if r.status_code == 200:
                j = r.json()
                ch = j.get('choices') or []
                if not ch:
                    last = f'HTTP 200 but no choices: {str(j)[:200]}'
                    if verbose: print('   ', last)
                    return None, {'error': last}
                return ch[0]['message']['content'], j.get('usage', {})
            if r.status_code == 400 and _adapt(m, r.text):
                last = f'HTTP 400 adapted: {r.text[:120]}'
                continue
            last = f'HTTP {r.status_code}: {r.text[:250]}'
            if verbose: print(f'   attempt {i+1}: {last}')
            if r.status_code in (429, 500, 502, 503, 529):
                time.sleep(2 ** i + random.random()); continue
            return None, {'error': last}
        except requests.RequestException as ex:
            last = f'{type(ex).__name__}: {str(ex)[:250]}'
            if verbose: print(f'   attempt {i+1}: {last}')
            time.sleep(2 ** i + random.random())
    return None, {'error': f'exhausted retries; last failure -> {last}'}

def run_arm(m, task, arm_name, items, desc_key, gt_key):
    out = RESULTS / f"{m['id']}__{task}__{arm_name}.jsonl"
    done = set()
    if out.exists():
        with open(out, encoding='utf-8') as f:
            for line in f:
                try: done.add(json.loads(line)['cve'])
                except Exception: pass
    todo = [r for r in items if r.get('cve', r.get('URL')) not in done]
    print(f"{m['id']} {task} {arm_name}: {len(done)} done, {len(todo)} to go")
    if not todo: return out

    def work(r):
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        txt, usage = call(m, build_prompt(task, r[desc_key]))
        return {'cve': cve, 'gt': r[gt_key], 'response': txt,
                'usage': usage, 'model': m['id'], 'task': task, 'arm': arm_name}

    with open(out, 'a', encoding='utf-8') as f, ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = {ex.submit(work, r): r for r in todo}
        for n, fut in enumerate(as_completed(futs), 1):
            rec = fut.result()
            with _lock:
                f.write(json.dumps(rec) + '\n'); f.flush()
            if n % 25 == 0: print(f'   {n}/{len(todo)}')
    return out

### 5b. Discover exact model strings

Do not guess model identifiers - ask each provider. Model strings change between
releases and a wrong one fails silently as a 404 mid-sweep. Copy the exact ids you
want into `MODELS` below, then look up each cutoff on the provider's model card.

In [ ]:
import requests

def list_models(p, filt=''):
    try:
        r = requests.get(PROVIDERS[p]['base'] + '/models',
                         headers={'Authorization': 'Bearer ' + KEYS[p]}, timeout=60)
        if r.status_code != 200:
            print(f'{p}: HTTP {r.status_code} {r.text[:120]}'); return []
        ids = sorted(m['id'] for m in r.json().get('data', []))
        hits = [i for i in ids if filt.lower() in i.lower()]
        print(f'--- {p}: {len(ids)} models, {len(hits)} matching "{filt}" ---')
        for i in hits[:40]: print('   ', i)
        return ids
    except Exception as ex:
        print(f'{p}: {type(ex).__name__} {ex}'); return []

for p in ACTIVE:
    list_models(p, '')

--- openai: 125 models, 125 matching "" ---
    babbage-002
    chat-latest
    chatgpt-image-latest
    davinci-002
    gpt-3.5-turbo
    gpt-3.5-turbo-0125
    gpt-3.5-turbo-1106
    gpt-3.5-turbo-16k
    gpt-3.5-turbo-instruct
    gpt-3.5-turbo-instruct-0914
    gpt-4
    gpt-4-0613
    gpt-4-turbo
    gpt-4-turbo-2024-04-09
    gpt-4.1
    gpt-4.1-2025-04-14
    gpt-4.1-mini
    gpt-4.1-mini-2025-04-14
    gpt-4.1-nano
    gpt-4.1-nano-2025-04-14
    gpt-4o
    gpt-4o-2024-05-13
    gpt-4o-2024-08-06
    gpt-4o-2024-11-20
    gpt-4o-mini
    gpt-4o-mini-2024-07-18
    gpt-4o-mini-search-preview
    gpt-4o-mini-search-preview-2025-03-11
    gpt-4o-mini-transcribe
    gpt-4o-mini-transcribe-2025-03-20
    gpt-4o-mini-transcribe-2025-12-15
    gpt-4o-mini-tts
    gpt-4o-mini-tts-2025-03-20
    gpt-4o-mini-tts-2025-12-15
    gpt-4o-search-preview
    gpt-4o-search-preview-2025-03-11
    gpt-4o-transcribe
    gpt-4o-transcribe-diarize
    gpt-5
    gpt-5-2025-08-07


### 5c. Register the temporal arm

Selection criterion is **cutoff spread**, not popularity - each model's accuracy cliff
should sit at its own cutoff. Every `cutoff` needs a `cutoff_src` pointing at the
provider's own model card. Aggregator blogs are not citable.

In [ ]:
# Cutoffs harvested from developers.openai.com/api/docs/models/<id>.md
# Exposure status is relative to the benchmark's 2024 publication window:
#   gpt-4o  cutoff 2023-10-01 -> UNEXPOSED to the benchmark (the control)
#   gpt-4.1 cutoff 2024-06-01 -> exposed
#   gpt-5.5 cutoff 2025-12-01 -> exposed
MODELS = [
    {'id': 'gpt-4o',  'provider': 'openai', 'model': 'gpt-4o-2024-08-06',
     'cutoff': '2023-10-01',
     'cutoff_src': 'https://developers.openai.com/api/docs/models/gpt-4o'},
    {'id': 'gpt-4.1', 'provider': 'openai', 'model': 'gpt-4.1-2025-04-14',
     'cutoff': '2024-06-01',
     'cutoff_src': 'https://developers.openai.com/api/docs/models/gpt-4.1'},
    {'id': 'gpt-5.5', 'provider': 'openai', 'model': 'gpt-5.5-2026-04-23',
     'cutoff': '2025-12-01',
     'cutoff_src': 'https://developers.openai.com/api/docs/models/gpt-5.5'},
]

assert MODELS, 'Register at least one model before running the pilot.'
for m in MODELS:
    assert m['provider'] in ACTIVE, f"{m['id']}: provider not in ACTIVE"
    assert m.get('cutoff_src'), f"{m['id']}: cutoff_src required - cite the model card"
    assert m['cutoff'] < CUTOFF_BOUNDARY, (
        f"{m['id']} cutoff {m['cutoff']} is not before boundary {CUTOFF_BOUNDARY}. "
        'Either drop the model or push CUTOFF_BOUNDARY later.')
print('registered:', [(m['id'], m['cutoff']) for m in MODELS])

registered: [('gpt-4o', '2023-10-01'), ('gpt-4.1', '2024-06-01'), ('gpt-5.5', '2025-12-01')]


### 5d. Smoke test - one call per model

Costs pennies. Verifies auth, the model string, and that a parseable answer comes back
before you launch hundreds of calls.

In [ ]:
# Raw connectivity + auth probe before per-model calls.
import requests as _rq
for _p in ACTIVE:
    _k = KEYS.get(_p, '')
    print(f"{_p}: key length {len(_k)}, starts {_k[:7]!r}" if _k else f'{_p}: NO KEY')
    try:
        _r = _rq.get(PROVIDERS[_p]['base'] + '/models',
                     headers={'Authorization': 'Bearer ' + _k}, timeout=60)
        print(f'   GET /models -> HTTP {_r.status_code} {_r.text[:120] if _r.status_code != 200 else "OK"}')
    except Exception as _e:
        print(f'   GET /models -> {type(_e).__name__}: {_e}')

probe = post_arm[0]
failed = []
for m in MODELS:
    txt, usage = call(m, build_prompt('rcm', probe['description']), verbose=True)
    ok = 'OK ' if txt else 'FAIL'
    print(f"{ok} {m['id']:12s} -> {str(txt)[-60:]!r}")
    if not txt:
        failed.append((m['id'], usage.get('error', '')))
        print(f"      {usage.get('error','')[:200]}")

assert not failed, (
    'Smoke test failed for: ' + ', '.join(f[0] for f in failed) +
    '. Fix before running section 6, or you will burn calls that return nothing.')
print('all models responding')

openai: key length 164, starts 'sk-proj'
   GET /models -> HTTP 200 OK
OK  gpt-4o       -> 'ch aligns with the nature of clickjacking attacks.\n\nCWE-1021'
OK  gpt-4.1      -> ', allowing attackers to overlay malicious content.\n\nCWE-1021'
   [adapt] gpt-5.5 -> tokens=max_completion_tokens temperature=on
   [adapt] gpt-5.5 -> tokens=max_completion_tokens temperature=off
OK  gpt-5.5      -> 'standard CWE category for clickjacking weaknesses.\n\nCWE-1021'
all models responding


## 6. Run the pilot

If a model previously failed, its checkpoint file holds null responses and would be
treated as already done. Purge those first.

In [ ]:
# Delete checkpoints whose responses are all null (from a failed run).
for p in sorted(RESULTS.glob('*.jsonl')):
    with open(p, encoding='utf-8') as f:
        recs = [json.loads(l) for l in f if l.strip()]
    if recs and all(r.get('response') is None for r in recs):
        print(f'purging {p.name} ({len(recs)} null records)')
        p.unlink()
print('checkpoint check done')

checkpoint check done


## 6. Run the pilot

In [ ]:
for m in MODELS:
    for task in TASKS:
        gt_post = 'GT_cwe' if task == 'rcm' else 'cvss_vector'
        run_arm(m, task, 'post', post_arm, 'description', gt_post)
        run_arm(m, task, 'bench', bench_arm, 'Description', 'GT')
print('pilot complete')

gpt-4o rcm post: 914 done, 870 to go
   25/870
   50/870
   75/870
   100/870
   125/870
   150/870
   175/870
   200/870
   225/870
   250/870
   275/870
   300/870
   325/870
   350/870
   375/870
   400/870
   425/870
   450/870
   475/870
   500/870
   525/870
   550/870
   575/870
   600/870
   625/870
   650/870
   675/870
   700/870
   725/870
   750/870
   775/870
   800/870
   825/870
   850/870
gpt-4o rcm bench: 676 done, 997 to go
   25/997
   50/997
   75/997
   100/997
   125/997
   150/997
   175/997
   200/997
   225/997
   250/997
   275/997
   300/997
   325/997
   350/997
   375/997
   400/997
   425/997
   450/997
   475/997
   500/997
   525/997
   550/997
   575/997
   600/997
   625/997
   650/997
   675/997
   700/997
   725/997
   750/997
   775/997
   800/997
   825/997
   850/997
   875/997
   900/997
   925/997
   950/997
   975/997
gpt-4.1 rcm post: 914 done, 870 to go
   25/870
   50/870
   75/870
   100/870
   125/870
   150/870
   175/870
   200/870
   22

## 7. Grade and test

RCM: the prompt instructs the model to end with only the CWE ID, so parse the last
line first and fall back to the final CWE mention. Unparseable responses are counted
as wrong AND reported separately - a parse-rate difference between arms is itself a result.

In [ ]:
CWE_RE = re.compile(r'CWE-\d+')

def parse_rcm(txt):
    if not txt: return None
    last = [l.strip() for l in txt.strip().split('\n') if l.strip()]
    if last:
        m = CWE_RE.fullmatch(last[-1].strip(' .*`'))
        if m: return m.group(0)
    hits = CWE_RE.findall(txt)
    return hits[-1] if hits else None

ARM_IDS = {
    'post':  [r['cve'] for r in post_arm],
    'bench': [re.search(r'CVE-\d{4}-\d+', r['URL']).group(0) for r in bench_arm],
}

def load(m_id, task, arm):
    # Restrict to the CURRENT arm and keep the LAST record per CVE. Guards against
    # checkpoints that accumulated records from an earlier, differently-drawn arm.
    p = RESULTS / f'{m_id}__{task}__{arm}.jsonl'
    if not p.exists(): return []
    want = set(ARM_IDS[arm])
    by_cve = {}
    extra = 0
    with open(p, encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            if r['cve'] in want:
                by_cve[r['cve']] = r
            else:
                extra += 1
    if extra:
        print(f'   [{m_id}/{arm}] ignored {extra} records not in the current arm')
    missing = len(want) - len(by_cve)
    if missing:
        print(f'   [{m_id}/{arm}] WARNING {missing} arm items have no response yet')
    return list(by_cve.values())

def grade_rcm(recs):
    n = ok = unparsed = 0
    for r in recs:
        n += 1
        p = parse_rcm(r['response'])
        if p is None: unparsed += 1
        elif p == r['gt']: ok += 1
    return n, ok, unparsed

import math
def two_prop(k1, n1, k2, n2):
    p1, p2 = k1/n1, k2/n2
    p = (k1+k2)/(n1+n2)
    se = math.sqrt(p*(1-p)*(1/n1+1/n2))
    z = (p1-p2)/se if se else 0.0
    from statistics import NormalDist
    return p1, p2, z, 2*(1-NormalDist().cdf(abs(z)))

for m in MODELS:
    for task in TASKS:
        if task != 'rcm': continue
        nb, kb, ub = grade_rcm(load(m['id'], task, 'bench'))
        np_, kp, up = grade_rcm(load(m['id'], task, 'post'))
        if not nb or not np_: continue
        p1, p2, z, pval = two_prop(kb, nb, kp, np_)
        print(f"\n=== {m['id']} / {task} ===")
        print(f'  benchmark arm : {100*p1:.1f}%  (n={nb}, unparsed={ub})')
        print(f'  post-cutoff   : {100*p2:.1f}%  (n={np_}, unparsed={up})')
        print(f'  gap           : {100*(p1-p2):+.1f} pp   z={z:.2f}  p={pval:.4g}')
        print(f'  vs TF-IDF gap : {100*(acc_b-acc_p):+.1f} pp  <- subtract this baseline')

   [gpt-4o/post] ignored 787 records not in the current arm

=== gpt-4o / rcm ===
  benchmark arm : 72.9%  (n=997, unparsed=34)
  post-cutoff   : 73.2%  (n=997, unparsed=30)
  gap           : -0.3 pp   z=-0.15  p=0.8796
  vs TF-IDF gap : -14.5 pp  <- subtract this baseline
   [gpt-4.1/post] ignored 787 records not in the current arm

=== gpt-4.1 / rcm ===
  benchmark arm : 73.2%  (n=997, unparsed=13)
  post-cutoff   : 72.2%  (n=997, unparsed=25)
  gap           : +1.0 pp   z=0.50  p=0.6151
  vs TF-IDF gap : -14.5 pp  <- subtract this baseline
   [gpt-5.5/post] ignored 540 records not in the current arm

=== gpt-5.5 / rcm ===
  benchmark arm : 74.1%  (n=997, unparsed=31)
  post-cutoff   : 76.5%  (n=997, unparsed=44)
  gap           : -2.4 pp   z=-1.25  p=0.2125
  vs TF-IDF gap : -14.5 pp  <- subtract this baseline


## Reading the pilot

- **LLM gap much larger than the TF-IDF gap** - memorization signal is real. Scale to
  N=999, add models, decide on dose-response.
- **LLM gap approximately equals the TF-IDF gap** - the drop is task difficulty, not
  memorization. Pivot the paper to identifier-stripping, which tests the mechanism directly.
- **No gap at all** - a real null. Publishable as a correction to the contamination
  assumption, but reframe the paper before writing.

Do not scale up until the pilot says which of these you are in.

In [ ]:
SAVE_BIG_POOLS = False   # True also pickles clean/post pools (~50MB, re-derivable)

import os, json, pickle, shutil, hashlib, platform, datetime as dt
from pathlib import Path

_PROJ = Path('/content/drive/MyDrive/ctibench-audit')
_STAMP = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
_ARCH = _PROJ / 'archive' / _STAMP
(_ARCH / 'results').mkdir(parents=True, exist_ok=True)
G = globals()

_SMALL = ['PILOT_N','TASKS','MAX_WORKERS','CUTOFF_BOUNDARY','SEED','SNAP',
          'MODELS','ACTIVE','PROVIDERS','PARAM_STYLE','CUTOFFS',
          'BENCH_HIST','BENCH_CWE_DIST','COMMON','EXCLUDED','target',
          'post_arm','bench_arm','ARM_IDS','rcm_bench','vsp_bench',
          'acc_b','acc_p','BENCH_LABELS','DUP_THRESHOLDS','audit',
          'RCM_PROMPT','VSP_HEAD','maj','spans']
_BIG = ['clean','post','post_pool_all','clean_new','clean_old','rows','pool']

saved, skipped = {}, []
for name in _SMALL + (_BIG if SAVE_BIG_POOLS else []):
    if name not in G:
        continue
    try:
        obj = G[name]
        if name in ('PROVIDERS',):
            obj = {k: {kk: vv for kk, vv in v.items() if 'key' not in kk.lower()}
                   for k, v in obj.items()}
        if isinstance(obj, Path):
            obj = str(obj)
        pickle.dumps(obj)
        saved[name] = obj
    except Exception as e:
        skipped.append(f'{name}: {type(e).__name__}')

with open(_ARCH / 'session.pkl', 'wb') as f:
    pickle.dump(saved, f, protocol=4)

_RES = _PROJ / 'results'
n_files = n_recs = 0
if _RES.exists():
    for p in sorted(_RES.glob('*.jsonl')):
        shutil.copy2(p, _ARCH / 'results' / p.name)
        n_files += 1
        n_recs += sum(1 for _ in open(p, encoding='utf-8'))

for sub in ('derived',):
    d = _PROJ / sub
    if d.exists():
        (_ARCH / sub).mkdir(exist_ok=True)
        for p in d.glob('*.json'):
            shutil.copy2(p, _ARCH / sub / p.name)

import re as _re
_CWE = _re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

summary = {}
if 'ARM_IDS' in G and 'MODELS' in G:
    for m in G['MODELS']:
        for arm in ('bench', 'post'):
            p = _RES / f"{m['id']}__rcm__{arm}.jsonl"
            if not p.exists(): continue
            want = set(G['ARM_IDS'][arm]); by = {}
            for line in open(p, encoding='utf-8'):
                if not line.strip(): continue
                r = json.loads(line)
                if r['cve'] in want: by[r['cve']] = r
            n = len(by)
            ok = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
            un = sum(1 for r in by.values() if _parse(r['response']) is None)
            tin = sum((r.get('usage') or {}).get('prompt_tokens', 0) for r in by.values())
            tout = sum((r.get('usage') or {}).get('completion_tokens', 0) for r in by.values())
            summary[f"{m['id']}/{arm}"] = {'n': n, 'correct': ok,
                'acc': round(100*ok/n, 2) if n else None, 'unparsed': un,
                'tokens_in': tin, 'tokens_out': tout}

manifest = {
    'archived_at': dt.datetime.now().isoformat(),
    'stamp': _STAMP,
    'config': {k: str(G.get(k)) for k in
               ('PILOT_N','CUTOFF_BOUNDARY','SEED','TASKS','SNAP')},
    'models': [{k: v for k, v in m.items()} for m in G.get('MODELS', [])],
    'arms': {'post': len(G.get('post_arm', [])), 'bench': len(G.get('bench_arm', []))},
    'tfidf_stage1_confounded': {'bench': G.get('acc_b'), 'post': G.get('acc_p')},
    'results': summary,
    'files': {'response_files': n_files, 'response_records': n_recs},
    'pickled_vars': sorted(saved), 'skipped_vars': skipped,
    'env': {'python': platform.python_version()},
}
try:
    import sklearn, numpy
    manifest['env'].update({'sklearn': sklearn.__version__, 'numpy': numpy.__version__})
except Exception:
    pass
(_ARCH / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

lines = []
for p in sorted(_ARCH.rglob('*')):
    if p.is_file() and p.name != 'CHECKSUMS.txt':
        h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        lines.append(f'{h}  {p.relative_to(_ARCH)}  {p.stat().st_size}')
(_ARCH / 'CHECKSUMS.txt').write_text('\n'.join(lines))

print(f'ARCHIVE -> {_ARCH}')
print(f'  pickled vars   : {len(saved)}  ({", ".join(sorted(saved)[:8])}...)')
if skipped: print(f'  skipped        : {skipped}')
print(f'  response files : {n_files}  ({n_recs} records)')
print(f'  total size     : {sum(p.stat().st_size for p in _ARCH.rglob("*") if p.is_file())/1e6:.1f} MB')
print()
print(f'{"model/arm":22s}{"n":>6s}{"acc%":>8s}{"unpars":>8s}{"tok_in":>10s}{"tok_out":>10s}')
for k, v in summary.items():
    print(f'{k:22s}{v["n"]:6d}{v["acc"]:8.1f}{v["unparsed"]:8d}'
          f'{v["tokens_in"]:10d}{v["tokens_out"]:10d}')
print()
print('reload later with:')
print(f"  import pickle; S = pickle.load(open('{_ARCH}/session.pkl','rb'))")
print("  post_arm = S['post_arm']; bench_arm = S['bench_arm']   # etc.")

ARCHIVE -> /content/drive/MyDrive/ctibench-audit/archive/20260906_070334
  pickled vars   : 26  (ACTIVE, ARM_IDS, BENCH_HIST, BENCH_LABELS, COMMON, CUTOFF_BOUNDARY, EXCLUDED, MAX_WORKERS...)
  response files : 6  (11726 records)
  total size     : 15.2 MB

model/arm                  n    acc%  unpars    tok_in   tok_out
gpt-4o/bench             997    72.9      34    116643    153810
gpt-4o/post              997    73.2      30    126900    158540
gpt-4.1/bench            997    73.2      13    121236    116494
gpt-4.1/post             997    72.2      25    128064    121935
gpt-5.5/bench            997    74.1      31    131154    194516
gpt-5.5/post             997    76.5      44    135520    212695

reload later with:
  import pickle; S = pickle.load(open('/content/drive/MyDrive/ctibench-audit/archive/20260906_070334/session.pkl','rb'))
  post_arm = S['post_arm']; bench_arm = S['bench_arm']   # etc.


In [ ]:
import json, re, random
import numpy as np
from pathlib import Path
from itertools import combinations
from math import comb

N_BOOT = 10000
random.seed(20261005); np.random.seed(20261005)
_PROJ = Path('/content/drive/MyDrive/ctibench-audit')
_RES  = _PROJ / 'results'

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def vec(mid, arm):
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    want = set(ARM_IDS[arm]); by = {}
    for line in open(p, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r['cve'] in want:
                by[r['cve']] = int(_parse(r['response']) == r['gt'])
    return by

IDS = {a: list(ARM_IDS[a]) for a in ('bench', 'post')}
V = {}
for m in MODELS:
    for arm in ('bench', 'post'):
        d = vec(m['id'], arm)
        V[(m['id'], arm)] = np.array([d.get(c, 0) for c in IDS[arm]], dtype=float)

n_b, n_p = len(IDS['bench']), len(IDS['post'])
print(f'items: bench={n_b}, post={n_p} | bootstrap B={N_BOOT}\n')

idx_b = np.random.randint(0, n_b, size=(N_BOOT, n_b))
idx_p = np.random.randint(0, n_p, size=(N_BOOT, n_p))
boot = {}
for m in MODELS:
    boot[(m['id'],'bench')] = V[(m['id'],'bench')][idx_b].mean(axis=1)*100
    boot[(m['id'],'post')]  = V[(m['id'],'post')][idx_p].mean(axis=1)*100

def ci(a, lo=2.5, hi=97.5):
    return np.percentile(a, lo), np.percentile(a, hi)

print(f'{"model":10s}{"arm":7s}{"acc":>7s}{"95% CI":>18s}')
for m in MODELS:
    for arm in ('bench','post'):
        pt = V[(m['id'],arm)].mean()*100; lo,hi = ci(boot[(m['id'],arm)])
        print(f'{m["id"]:10s}{arm:7s}{pt:7.1f}   [{lo:5.1f}, {hi:5.1f}]')

print(f'\n{"model":10s}{"gap":>7s}{"95% CI":>18s}{"incl. 0?":>10s}')
for m in MODELS:
    g = boot[(m['id'],'bench')] - boot[(m['id'],'post')]
    pt = V[(m['id'],'bench')].mean()*100 - V[(m['id'],'post')].mean()*100
    lo,hi = ci(g)
    print(f'{m["id"]:10s}{pt:7.1f}   [{lo:5.1f}, {hi:5.1f}]{("yes" if lo<0<hi else "NO"):>10s}')

print('\nPAIRED model differences within each arm (same items):')
print(f'{"arm":7s}{"pair":24s}{"diff":>7s}{"95% CI":>18s}{"McNemar p":>11s}')
for arm in ('bench','post'):
    idx = idx_b if arm=='bench' else idx_p
    for a,b in combinations([m['id'] for m in MODELS], 2):
        va, vb = V[(a,arm)], V[(b,arm)]
        bd = (va-vb)[idx].mean(axis=1)*100
        lo,hi = ci(bd)
        n01 = int(((va==0)&(vb==1)).sum()); n10 = int(((va==1)&(vb==0)).sum())
        n = n01+n10
        p = 1.0 if n==0 else min(1.0, 2*sum(comb(n,k) for k in range(0,min(n01,n10)+1))/(2**n))
        flag = '' if lo<0<hi else '  *'
        print(f'{arm:7s}{a+" vs "+b:24s}{(va-vb).mean()*100:7.1f}   [{lo:5.1f}, {hi:5.1f}]{p:11.3f}{flag}')

print('\nEquivalence bounds on the benchmark arm (what we can exclude):')
for a,b in combinations([m['id'] for m in MODELS], 2):
    bd = (V[(a,'bench')]-V[(b,'bench')])[idx_b].mean(axis=1)*100
    lo,hi = ci(bd)
    print(f'  {a} vs {b}: |diff| < {max(abs(lo),abs(hi)):.1f} pp with 95% confidence')

print('\narm       spread            95% CI')
sp = {}
for arm in ('bench','post'):
    stack = np.vstack([boot[(m['id'],arm)] for m in MODELS])
    sp[arm] = stack.max(axis=0) - stack.min(axis=0)
    pt = max(V[(m['id'],arm)].mean() for m in MODELS)*100 - \
         min(V[(m['id'],arm)].mean() for m in MODELS)*100
    lo,hi = ci(sp[arm])
    print(f'{arm:8s}{pt:8.1f}   [{lo:5.1f}, {hi:5.1f}]')
d = sp['post'] - sp['bench']; lo,hi = ci(d)
print(f'\npost spread - bench spread: {d.mean():+.1f}  95% CI [{lo:+.1f}, {hi:+.1f}]')
print(f'P(post spread > bench spread) = {(d>0).mean():.3f}')

ctrl = MODELS[0]['id']
print(f'\nDiD vs control ({ctrl}):')
gc = boot[(ctrl,'bench')] - boot[(ctrl,'post')]
for m in MODELS[1:]:
    dd = (boot[(m['id'],'bench')] - boot[(m['id'],'post')]) - gc
    lo,hi = ci(dd)
    print(f'  {m["id"]:10s}{dd.mean():+6.1f}   95% CI [{lo:+5.1f}, {hi:+5.1f}]'
          f'   {"includes 0" if lo<0<hi else "EXCLUDES 0"}')

se = gc.std()
print(f'\nbootstrap SE of a gap: {se:.2f} pp -> ~{1.96*se:.1f} pp detectable at 95%')

items: bench=997, post=997 | bootstrap B=10000

model     arm        acc            95% CI
gpt-4o    bench     72.9   [ 70.1,  75.6]
gpt-4o    post      73.2   [ 70.4,  75.9]
gpt-4.1   bench     73.2   [ 70.4,  75.9]
gpt-4.1   post      72.2   [ 69.4,  74.9]
gpt-5.5   bench     74.1   [ 71.4,  76.7]
gpt-5.5   post      76.5   [ 73.8,  79.0]

model         gap            95% CI  incl. 0?
gpt-4o       -0.3   [ -4.2,   3.6]       yes
gpt-4.1       1.0   [ -2.9,   4.9]       yes
gpt-5.5      -2.4   [ -6.2,   1.4]       yes

PAIRED model differences within each arm (same items):
arm    pair                       diff            95% CI  McNemar p
bench  gpt-4o vs gpt-4.1          -0.3   [ -2.0,   1.4]      0.818
bench  gpt-4o vs gpt-5.5          -1.2   [ -3.2,   0.8]      0.276
bench  gpt-4.1 vs gpt-5.5         -0.9   [ -2.7,   0.9]      0.386
post   gpt-4o vs gpt-4.1           1.0   [ -1.1,   3.1]      0.403
post   gpt-4o vs gpt-5.5          -3.3   [ -5.7,  -1.0]      0.006  *
post   gpt-4.

In [ ]:
# ============================================================================
# ADD NON-OPENAI MODELS  (standalone)
# Requires from earlier cells: post_arm, bench_arm, ARM_IDS
# ============================================================================

# ---- 1. FILL THESE IN from each vendor's own model card ---------------------
NEW_MODELS = [
    {'id': 'llama-4', 'model': 'meta-llama/llama-4-maverick',
     'cutoff': '2024-08-31',
     'cutoff_src': 'https://www.llama.com/docs/model-cards-and-prompt-formats/llama4/'},
    {'id': 'claude',  'model': 'anthropic/claude-sonnet-4.5',
     'cutoff': '2025-07-01',
     'cutoff_src': 'https://docs.claude.com/en/docs/about-claude/models/overview'},
]
OR_BASE = 'https://openrouter.ai/api/v1'
BOUNDARY = globals().get('CUTOFF_BOUNDARY', '2026-06-01')

import json, re, time, random, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

_PROJ = Path('/content/drive/MyDrive/ctibench-audit')
_RES = _PROJ / 'results'; _RES.mkdir(parents=True, exist_ok=True)

for m in NEW_MODELS:
    assert m['cutoff'] != 'YYYY-MM-DD', f"{m['id']}: fill in the real cutoff"
    assert m['cutoff_src'], f"{m['id']}: cutoff_src required (cite the model card)"
    assert m['cutoff'] < BOUNDARY, (
        f"{m['id']} cutoff {m['cutoff']} is not before boundary {BOUNDARY}. "
        "Pick an earlier model from the OpenRouter list.")

try:
    from google.colab import userdata
    OR_KEY = (userdata.get('OPENROUTER_API_KEY') or '').strip()
except Exception:
    OR_KEY = ''
if not OR_KEY:
    from getpass import getpass
    OR_KEY = getpass('OPENROUTER_API_KEY: ').strip()
assert OR_KEY, 'no OpenRouter key'

_STYLE = {}
def _body(m, prompt):
    st = _STYLE.get(m['id'], {'tok': 'max_tokens', 'temp': True})
    b = {'model': m['model'], 'messages': [{'role': 'user', 'content': prompt}],
         st['tok']: 600}
    if st['temp']: b['temperature'] = 0
    return b

def _adapt(m, err):
    st = _STYLE.get(m['id'], {'tok': 'max_tokens', 'temp': True}); ch = False
    if 'max_completion_tokens' in err and st['tok'] == 'max_tokens':
        st['tok'] = 'max_completion_tokens'; ch = True
    if 'temperature' in err and st['temp']:
        st['temp'] = False; ch = True
    if ch: _STYLE[m['id']] = st; print(f"   [adapt] {m['id']} -> {st}")
    return ch

def _call(m, prompt, tries=4, verbose=False):
    hdr = {'Authorization': 'Bearer ' + OR_KEY, 'Content-Type': 'application/json'}
    last = 'no attempt'
    for i in range(tries):
        try:
            r = requests.post(OR_BASE + '/chat/completions',
                              json=_body(m, prompt), headers=hdr, timeout=120)
            if r.status_code == 200:
                j = r.json(); ch = j.get('choices') or []
                if not ch:
                    last = f'200 but no choices: {str(j)[:200]}'
                    if verbose: print('   ', last)
                    return None, {'error': last}
                return ch[0]['message']['content'], j.get('usage', {})
            if r.status_code == 400 and _adapt(m, r.text):
                continue
            last = f'HTTP {r.status_code}: {r.text[:250]}'
            if verbose: print(f'   attempt {i+1}: {last}')
            if r.status_code in (429, 500, 502, 503, 529):
                time.sleep(2 ** i + random.random()); continue
            return None, {'error': last}
        except requests.RequestException as ex:
            last = f'{type(ex).__name__}: {str(ex)[:200]}'
            if verbose: print(f'   attempt {i+1}: {last}')
            time.sleep(2 ** i + random.random())
    return None, {'error': f'exhausted retries; last -> {last}'}

RCM = ('Analyze the following CVE description and map it to the appropriate CWE. '
       'Provide a brief justification for your choice. Ensure the last line of your '
       'response contains only the CWE ID.  CVE Description: ')

print('--- smoke test ---')
bad = []
for m in NEW_MODELS:
    txt, u = _call(m, RCM + post_arm[0]['description'] + ' ', verbose=True)
    print(('OK   ' if txt else 'FAIL ') + f"{m['id']:10s} {str(txt)[-55:]!r}")
    if not txt: bad.append((m['id'], u.get('error', '')))
assert not bad, f'smoke test failed: {bad}'

_lock = threading.Lock()
def run(m, arm, items, dkey, gkey):
    out = _RES / f"{m['id']}__rcm__{arm}.jsonl"
    done = set()
    if out.exists():
        for line in open(out, encoding='utf-8'):
            if line.strip():
                try: done.add(json.loads(line)['cve'])
                except Exception: pass
    todo = []
    for r in items:
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        if cve not in done: todo.append((cve, r))
    print(f"{m['id']} {arm}: {len(done)} done, {len(todo)} to go")
    if not todo: return
    def work(pair):
        cve, r = pair
        txt, u = _call(m, RCM + r[dkey] + ' ')
        return {'cve': cve, 'gt': r[gkey], 'response': txt, 'usage': u,
                'model': m['id'], 'task': 'rcm', 'arm': arm}
    with open(out, 'a', encoding='utf-8') as f, ThreadPoolExecutor(6) as ex:
        for n, fut in enumerate(as_completed([ex.submit(work, p) for p in todo]), 1):
            rec = fut.result()
            with _lock: f.write(json.dumps(rec) + '\n'); f.flush()
            if n % 50 == 0: print(f'   {n}/{len(todo)}')

print('\n--- running ---')
for m in NEW_MODELS:
    run(m, 'post',  post_arm,  'description', 'GT_cwe')
    run(m, 'bench', bench_arm, 'Description', 'GT')
print('done')

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        mm = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if mm: return mm.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def acc(mid, arm):
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    if not p.exists(): return None, 0, 0
    want = set(ARM_IDS[arm]); by = {}
    for line in open(p, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r['cve'] in want: by[r['cve']] = r
    n = len(by)
    ok = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
    un = sum(1 for r in by.values() if _parse(r['response']) is None)
    return (100*ok/n if n else None), n, un

all_ids = [m['id'] for m in globals().get('MODELS', [])] + [m['id'] for m in NEW_MODELS]
print(f'\n{"model":12s}{"bench":>9s}{"post":>9s}{"n":>7s}{"unpars":>8s}')
rows = {}
for mid in all_ids:
    ab, nb, ub = acc(mid, 'bench'); ap, npp, up = acc(mid, 'post')
    if ab is None: continue
    rows[mid] = (ab, ap)
    print(f'{mid:12s}{ab:9.1f}{ap:9.1f}{nb:7d}{ub+up:8d}')

if rows:
    bs = [v[0] for v in rows.values()]; ps = [v[1] for v in rows.values()]
    print(f'\nbenchmark spread across {len(rows)} models: {max(bs)-min(bs):.1f} pp')
    print(f'post-cutoff spread:                    {max(ps)-min(ps):.1f} pp')

OPENROUTER_API_KEY: ··········
--- smoke test ---
OK   llama-4    'making it vulnerable to clickjacking attacks.\n\nCWE-1021'
OK   claude     'and trick users into unintended interactions.\n\nCWE-1021'

--- running ---
llama-4 post: 0 done, 997 to go
   50/997
   100/997
   150/997
   200/997
   250/997
   300/997
   350/997
   400/997
   450/997
   500/997
   550/997
   600/997
   650/997
   700/997
   750/997
   800/997
   850/997
   900/997
   950/997
llama-4 bench: 0 done, 997 to go
   50/997
   100/997
   150/997
   200/997
   250/997
   300/997
   350/997
   400/997
   450/997
   500/997
   550/997
   600/997
   650/997
   700/997
   750/997
   800/997
   850/997
   900/997
   950/997
claude post: 0 done, 997 to go
   50/997
   100/997
   150/997
   200/997
   250/997
   300/997
   350/997
   400/997
   450/997
   500/997
   550/997
   600/997
   650/997
   700/997
   750/997
   800/997
   850/997
   900/997
   950/997
claude bench: 0 done, 997 to go
   50/997
   100/997
   150/9

In [ ]:
SAVE_BIG_POOLS = False

import os, json, pickle, shutil, hashlib, platform, re
import datetime as dt
from pathlib import Path

_PROJ  = Path('/content/drive/MyDrive/ctibench-audit')
_RES   = _PROJ / 'results'
_STAMP = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
_ARCH  = _PROJ / 'archive' / _STAMP
(_ARCH / 'results').mkdir(parents=True, exist_ok=True)
G = globals()

_SMALL = ['PILOT_N','TASKS','MAX_WORKERS','CUTOFF_BOUNDARY','SEED','SNAP',
          'MODELS','NEW_MODELS','EXTRA_MODELS','ACTIVE','PROVIDERS',
          'PARAM_STYLE','_STYLE','CUTOFFS','BENCH_HIST','BENCH_CWE_DIST',
          'COMMON','EXCLUDED','target','post_arm','bench_arm','ARM_IDS',
          'rcm_bench','vsp_bench','acc_b','acc_p','BENCH_LABELS',
          'DUP_THRESHOLDS','audit','RCM_PROMPT','VSP_HEAD','RCM','maj',
          'spans','boot','V','summary']
_BIG = ['clean','post','post_pool_all','clean_new','clean_old','rows','pool']

saved, skipped = {}, []
for name in _SMALL + (_BIG if SAVE_BIG_POOLS else []):
    if name not in G:
        continue
    try:
        obj = G[name]
        if name == 'PROVIDERS':
            obj = {k: {kk: vv for kk, vv in v.items() if 'key' not in kk.lower()}
                   for k, v in obj.items()}
        if isinstance(obj, Path):
            obj = str(obj)
        pickle.dumps(obj)
        saved[name] = obj
    except Exception as e:
        skipped.append(f'{name}: {type(e).__name__}')

with open(_ARCH / 'session.pkl', 'wb') as f:
    pickle.dump(saved, f, protocol=4)

n_files = n_recs = 0
if _RES.exists():
    for p in sorted(_RES.glob('*.jsonl')):
        shutil.copy2(p, _ARCH / 'results' / p.name)
        n_files += 1
        n_recs += sum(1 for _ in open(p, encoding='utf-8'))
d = _PROJ / 'derived'
if d.exists():
    (_ARCH / 'derived').mkdir(exist_ok=True)
    for p in d.glob('*.json'):
        shutil.copy2(p, _ARCH / 'derived' / p.name)

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

ALL = ([m['id'] for m in G.get('MODELS', [])] +
       [m['id'] for m in G.get('NEW_MODELS', [])] +
       [m['id'] for m in G.get('EXTRA_MODELS', [])])
ALL = list(dict.fromkeys(ALL))

summary = {}
if 'ARM_IDS' in G:
    for mid in ALL:
        for arm in ('bench', 'post'):
            p = _RES / f'{mid}__rcm__{arm}.jsonl'
            if not p.exists(): continue
            want = set(G['ARM_IDS'][arm]); by = {}; stale = 0
            for line in open(p, encoding='utf-8'):
                if not line.strip(): continue
                r = json.loads(line)
                if r['cve'] in want: by[r['cve']] = r
                else: stale += 1
            n = len(by)
            got = sum(1 for r in by.values() if r.get('response'))
            ok  = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
            un  = sum(1 for r in by.values() if _parse(r['response']) is None)
            tin = sum((r.get('usage') or {}).get('prompt_tokens', 0) for r in by.values())
            tout= sum((r.get('usage') or {}).get('completion_tokens', 0) for r in by.values())
            summary[f'{mid}/{arm}'] = {
                'n': n, 'target': len(want), 'answered': got,
                'complete': n == len(want) and got == n,
                'acc': round(100*ok/n, 2) if n else None,
                'unparsed': un, 'stale_ignored': stale,
                'tokens_in': tin, 'tokens_out': tout}

manifest = {
    'archived_at': dt.datetime.now().isoformat(), 'stamp': _STAMP,
    'config': {k: str(G.get(k)) for k in
               ('PILOT_N','CUTOFF_BOUNDARY','SEED','TASKS','SNAP')},
    'models': G.get('MODELS', []), 'new_models': G.get('NEW_MODELS', []),
    'extra_models': G.get('EXTRA_MODELS', []),
    'arms': {'post': len(G.get('post_arm', [])),
             'bench': len(G.get('bench_arm', []))},
    'results': summary,
    'files': {'response_files': n_files, 'response_records': n_recs},
    'pickled_vars': sorted(saved), 'skipped_vars': skipped,
    'env': {'python': platform.python_version()},
}
try:
    import sklearn, numpy
    manifest['env'].update({'sklearn': sklearn.__version__,
                            'numpy': numpy.__version__})
except Exception:
    pass
(_ARCH / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

lines = []
for p in sorted(_ARCH.rglob('*')):
    if p.is_file() and p.name != 'CHECKSUMS.txt':
        lines.append(f'{hashlib.sha256(p.read_bytes()).hexdigest()[:16]}  '
                     f'{p.relative_to(_ARCH)}  {p.stat().st_size}')
(_ARCH / 'CHECKSUMS.txt').write_text('\n'.join(lines))

size = sum(p.stat().st_size for p in _ARCH.rglob('*') if p.is_file())
print(f'ARCHIVE -> {_ARCH}')
print(f'  pickled {len(saved)} vars | {n_files} response files '
      f'({n_recs} records) | {size/1e6:.1f} MB')
if skipped: print(f'  skipped: {skipped}')
print()
print(f'{"model/arm":20s}{"have":>7s}{"want":>7s}{"acc%":>8s}{"unp":>6s}{"status":>12s}')
incomplete = []
for k, v in summary.items():
    a = f'{v["acc"]:8.1f}' if v['acc'] is not None else f'{"-":>8s}'
    st = 'complete' if v['complete'] else 'PARTIAL'
    if not v['complete']: incomplete.append(k)
    print(f'{k:20s}{v["answered"]:7d}{v["target"]:7d}{a}{v["unparsed"]:6d}{st:>12s}')
print()
if incomplete:
    print(f'PARTIAL: {", ".join(incomplete)}')
    print('  -> safe. Re-run the model cell; it resumes from the JSONL.')
else:
    print('all arms complete')
print()
print('restore with:')
print(f"  import pickle; S = pickle.load(open('{_ARCH}/session.pkl','rb'))")

ARCHIVE -> /content/drive/MyDrive/ctibench-audit/archive/20260906_085001
  pickled 32 vars | 10 response files (15714 records) | 21.7 MB

model/arm              have   want    acc%   unp      status
gpt-4o/bench            963    997    72.9    34     PARTIAL
gpt-4o/post             967    997    73.2    30     PARTIAL
gpt-4.1/bench           984    997    73.2    13     PARTIAL
gpt-4.1/post            972    997    72.2    25     PARTIAL
gpt-5.5/bench           966    997    74.1    31     PARTIAL
gpt-5.5/post            953    997    76.5    44     PARTIAL
llama-4/bench           997    997    74.6     1    complete
llama-4/post            997    997    74.4     0    complete
claude/bench            629    997    47.6   368     PARTIAL
claude/post             648    997    50.1   349     PARTIAL

PARTIAL: gpt-4o/bench, gpt-4o/post, gpt-4.1/bench, gpt-4.1/post, gpt-5.5/bench, gpt-5.5/post, claude/bench, claude/post
  -> safe. Re-run the model cell; it resumes from the JSONL.

restore 

In [ ]:
import json, re
from pathlib import Path
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
_CWE = re.compile(r'CWE-\d+')

recs = [json.loads(l) for l in open(_RES/'claude__rcm__bench.jsonl', encoding='utf-8') if l.strip()]
print(f'records: {len(recs)}')

def parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

fails = [r for r in recs if parse(r['response']) is None]
print(f'unparsed: {len(fails)}')

null   = sum(1 for r in fails if not r.get('response'))
nocwe  = sum(1 for r in fails if r.get('response') and not _CWE.search(r['response']))
hascwe = sum(1 for r in fails if r.get('response') and _CWE.search(r['response']))
print(f'  null/error responses : {null}')
print(f'  text but NO CWE at all: {nocwe}   <- truncation')
print(f'  CWE present, parser missed it: {hascwe}   <- parser bug')

toks = [(r.get("usage") or {}).get("completion_tokens", 0) for r in recs]
at_cap = sum(1 for t in toks if t >= 595)
print(f'\ncompletion tokens: mean {sum(toks)/len(toks):.0f}, max {max(toks)}')
print(f'responses at the 600 cap: {at_cap}  <- if high, truncation confirmed')

print('\n--- three failed responses, last 300 chars ---')
for r in fails[:3]:
    print(f'\n[{r["cve"]}] tokens={(r.get("usage") or {}).get("completion_tokens")}')
    print(repr(str(r['response'])[-300:]))

records: 997
unparsed: 368
  null/error responses : 368
  text but NO CWE at all: 0   <- truncation
  CWE present, parser missed it: 0   <- parser bug

completion tokens: mean 192, max 490
responses at the 600 cap: 0  <- if high, truncation confirmed

--- three failed responses, last 300 chars ---

[CVE-2024-0669] tokens=None
'None'

[CVE-2023-5905] tokens=None
'None'

[CVE-2024-0237] tokens=None
'None'


In [ ]:
import json, collections
from pathlib import Path
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
errs = collections.Counter()
for arm in ('bench','post'):
    for line in open(_RES/f'claude__rcm__{arm}.jsonl', encoding='utf-8'):
        if not line.strip(): continue
        r = json.loads(line)
        if not r.get('response'):
            errs[str((r.get('usage') or {}).get('error'))[:110]] += 1
for e, n in errs.most_common(10):
    print(f'{n:5d}  {e}')

  717  exhausted retries; last -> HTTP 429: {"error":{"message":"Rate limit exceeded: new-account-rpm/anthropic/claud


In [ ]:
import json
from pathlib import Path
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
for p in sorted(_RES.glob('*.jsonl')):
    recs = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
    good = [r for r in recs if r.get('response')]
    if len(good) != len(recs):
        with open(p, 'w', encoding='utf-8') as f:
            for r in good: f.write(json.dumps(r) + '\n')
        print(f'{p.name}: dropped {len(recs)-len(good)} nulls, kept {len(good)}')
print('done')

claude__rcm__bench.jsonl: dropped 368 nulls, kept 629
claude__rcm__post.jsonl: dropped 349 nulls, kept 648
gpt-4.1__rcm__bench.jsonl: dropped 56 nulls, kept 2261
gpt-4.1__rcm__post.jsonl: dropped 43 nulls, kept 1741
gpt-4o__rcm__bench.jsonl: dropped 85 nulls, kept 2232
gpt-4o__rcm__post.jsonl: dropped 55 nulls, kept 1729
gpt-5.5__rcm__bench.jsonl: dropped 70 nulls, kept 1917
gpt-5.5__rcm__post.jsonl: dropped 65 nulls, kept 1472
done


In [ ]:
def _call(m, prompt, tries=8, verbose=False):
    hdr = {'Authorization': 'Bearer ' + OR_KEY, 'Content-Type': 'application/json'}
    last = 'no attempt'
    for i in range(tries):
        try:
            r = requests.post(OR_BASE + '/chat/completions',
                              json=_body(m, prompt), headers=hdr, timeout=180)
            if r.status_code == 200:
                j = r.json(); ch = j.get('choices') or []
                if not ch:
                    return None, {'error': f'200 no choices: {str(j)[:200]}'}
                return ch[0]['message']['content'], j.get('usage', {})
            if r.status_code == 400 and _adapt(m, r.text):
                continue
            last = f'HTTP {r.status_code}: {r.text[:200]}'
            if r.status_code == 429:
                wait = float(r.headers.get('retry-after', 0)) or min(60, 5 * (i + 1))
                if verbose: print(f'   429, waiting {wait:.0f}s')
                time.sleep(wait); continue
            if r.status_code in (500, 502, 503, 529):
                time.sleep(2 ** i + random.random()); continue
            return None, {'error': last}
        except requests.RequestException as ex:
            last = f'{type(ex).__name__}: {str(ex)[:200]}'
            time.sleep(2 ** i + random.random())
    return None, {'error': f'exhausted retries; last -> {last}'}
print('_call patched: 8 tries, honours retry-after, linear backoff on 429')

_call patched: 8 tries, honours retry-after, linear backoff on 429


In [ ]:
NEW_MODELS = [m for m in NEW_MODELS if m['id'] == 'claude']

In [ ]:
CLAUDE = {'id': 'claude', 'model': 'anthropic/claude-sonnet-4.5',
          'cutoff': '2025-07-01',
          'cutoff_src': 'https://docs.claude.com/en/docs/about-claude/models/overview'}

WORKERS = 2          # drop to 1 if 429s persist
MAX_TRIES = 8

import json, re, time, random, threading, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

OR_BASE = 'https://openrouter.ai/api/v1'
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')

try:
    OR_KEY
except NameError:
    try:
        from google.colab import userdata
        OR_KEY = (userdata.get('OPENROUTER_API_KEY') or '').strip()
    except Exception:
        OR_KEY = ''
    if not OR_KEY:
        from getpass import getpass
        OR_KEY = getpass('OPENROUTER_API_KEY: ').strip()
assert OR_KEY, 'no OpenRouter key'

RCM = ('Analyze the following CVE description and map it to the appropriate CWE. '
       'Provide a brief justification for your choice. Ensure the last line of your '
       'response contains only the CWE ID.  CVE Description: ')

_hdr = {'Authorization': 'Bearer ' + OR_KEY, 'Content-Type': 'application/json'}
_429 = {'n': 0}

def call(prompt, verbose=False):
    body = {'model': CLAUDE['model'],
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0, 'max_tokens': 800}
    last = 'no attempt'
    for i in range(MAX_TRIES):
        try:
            r = requests.post(OR_BASE + '/chat/completions', json=body,
                              headers=_hdr, timeout=180)
            if r.status_code == 200:
                j = r.json(); ch = j.get('choices') or []
                if not ch:
                    return None, {'error': f'200 no choices: {str(j)[:200]}'}
                return ch[0]['message']['content'], j.get('usage', {})
            last = f'HTTP {r.status_code}: {r.text[:200]}'
            if r.status_code == 429:
                _429['n'] += 1
                wait = float(r.headers.get('retry-after', 0) or 0) or min(60, 5*(i+1))
                if verbose: print(f'   429 -> waiting {wait:.0f}s')
                time.sleep(wait); continue
            if r.status_code in (500, 502, 503, 529):
                time.sleep(2 ** i + random.random()); continue
            return None, {'error': last}
        except requests.RequestException as ex:
            last = f'{type(ex).__name__}: {str(ex)[:150]}'
            time.sleep(2 ** i + random.random())
    return None, {'error': f'exhausted retries; last -> {last}'}

print('--- smoke test ---')
_t, _u = call(RCM + post_arm[0]['description'] + ' ', verbose=True)
assert _t, f'smoke test failed: {_u}'
print(f'OK  claude -> {str(_t)[-55:]!r}')

_lock = threading.Lock()
def run_claude(arm, items, dkey, gkey):
    out = _RES / f'claude__rcm__{arm}.jsonl'
    done = set()
    if out.exists():
        for line in open(out, encoding='utf-8'):
            if line.strip():
                r = json.loads(line)
                if r.get('response'):
                    done.add(r['cve'])
    todo = []
    for r in items:
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        if cve not in done: todo.append((cve, r))
    print(f'\nclaude {arm}: {len(done)} done, {len(todo)} to go')
    if not todo: return
    def work(pair):
        cve, r = pair
        txt, u = call(RCM + r[dkey] + ' ')
        return {'cve': cve, 'gt': r[gkey], 'response': txt, 'usage': u,
                'model': 'claude', 'task': 'rcm', 'arm': arm}
    fails = 0
    with open(out, 'a', encoding='utf-8') as f, ThreadPoolExecutor(WORKERS) as ex:
        for n, fut in enumerate(as_completed([ex.submit(work, p) for p in todo]), 1):
            rec = fut.result()
            if not rec['response']: fails += 1
            with _lock: f.write(json.dumps(rec) + '\n'); f.flush()
            if n % 50 == 0:
                print(f'   {n}/{len(todo)}  (429s so far: {_429["n"]}, failed: {fails})')
    print(f'   finished: {len(todo)-fails} ok, {fails} failed')

run_claude('post',  post_arm,  'description', 'GT_cwe')
run_claude('bench', bench_arm, 'Description', 'GT')

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

print(f'\n{"arm":8s}{"n":>6s}{"answered":>10s}{"acc%":>8s}{"unparsed":>10s}')
for arm in ('bench', 'post'):
    p = _RES / f'claude__rcm__{arm}.jsonl'
    want = set(ARM_IDS[arm]); by = {}
    for line in open(p, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r['cve'] in want and r.get('response'): by[r['cve']] = r
    n = len(by)
    ok = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
    un = sum(1 for r in by.values() if _parse(r['response']) is None)
    print(f'{arm:8s}{len(want):6d}{n:10d}{100*ok/n if n else 0:8.1f}{un:10d}')
print(f'\ntotal 429s encountered: {_429["n"]}')
print('If "answered" is still below 997, re-run this cell -- it resumes.')

--- smoke test ---
OK  claude -> 'and trick users into unintended interactions.\n\nCWE-1021'

claude post: 648 done, 349 to go
   50/349  (429s so far: 0, failed: 0)
   100/349  (429s so far: 0, failed: 0)
   150/349  (429s so far: 0, failed: 0)
   200/349  (429s so far: 0, failed: 0)
   250/349  (429s so far: 0, failed: 0)
   300/349  (429s so far: 0, failed: 0)
   finished: 349 ok, 0 failed

claude bench: 629 done, 368 to go
   50/368  (429s so far: 0, failed: 0)
   100/368  (429s so far: 0, failed: 0)
   150/368  (429s so far: 0, failed: 0)
   200/368  (429s so far: 0, failed: 0)
   250/368  (429s so far: 0, failed: 0)
   300/368  (429s so far: 0, failed: 0)
   350/368  (429s so far: 0, failed: 0)
   finished: 368 ok, 0 failed

arm          n  answered    acc%  unparsed
bench      997       997    74.3         0
post       997       997    76.4         0

total 429s encountered: 0
If "answered" is still below 997, re-run this cell -- it resumes.


In [ ]:
NEW_MODELS = [
 {'id':'llama-4','model':'meta-llama/llama-4-maverick','cutoff':'2024-08-31',
  'cutoff_src':'https://www.llama.com/docs/model-cards-and-prompt-formats/llama4/'},
 {'id':'claude','model':'anthropic/claude-sonnet-4.5','cutoff':'2025-07-01',
  'cutoff_src':'https://docs.claude.com/en/docs/about-claude/models/overview'},
]

In [ ]:
SAVE_BIG_POOLS = False

import os, json, pickle, shutil, hashlib, platform, re
import datetime as dt
from pathlib import Path

_PROJ  = Path('/content/drive/MyDrive/ctibench-audit')
_RES   = _PROJ / 'results'
_STAMP = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
_ARCH  = _PROJ / 'archive' / _STAMP
(_ARCH / 'results').mkdir(parents=True, exist_ok=True)
G = globals()

_SMALL = ['PILOT_N','TASKS','MAX_WORKERS','CUTOFF_BOUNDARY','SEED','SNAP',
          'MODELS','NEW_MODELS','EXTRA_MODELS','ACTIVE','PROVIDERS',
          'PARAM_STYLE','_STYLE','CUTOFFS','BENCH_HIST','BENCH_CWE_DIST',
          'COMMON','EXCLUDED','target','post_arm','bench_arm','ARM_IDS',
          'rcm_bench','vsp_bench','acc_b','acc_p','BENCH_LABELS',
          'DUP_THRESHOLDS','audit','RCM_PROMPT','VSP_HEAD','RCM','maj',
          'spans','boot','V','summary']
_BIG = ['clean','post','post_pool_all','clean_new','clean_old','rows','pool']

saved, skipped = {}, []
for name in _SMALL + (_BIG if SAVE_BIG_POOLS else []):
    if name not in G:
        continue
    try:
        obj = G[name]
        if name == 'PROVIDERS':
            obj = {k: {kk: vv for kk, vv in v.items() if 'key' not in kk.lower()}
                   for k, v in obj.items()}
        if isinstance(obj, Path):
            obj = str(obj)
        pickle.dumps(obj)
        saved[name] = obj
    except Exception as e:
        skipped.append(f'{name}: {type(e).__name__}')

with open(_ARCH / 'session.pkl', 'wb') as f:
    pickle.dump(saved, f, protocol=4)

n_files = n_recs = 0
if _RES.exists():
    for p in sorted(_RES.glob('*.jsonl')):
        shutil.copy2(p, _ARCH / 'results' / p.name)
        n_files += 1
        n_recs += sum(1 for _ in open(p, encoding='utf-8'))
d = _PROJ / 'derived'
if d.exists():
    (_ARCH / 'derived').mkdir(exist_ok=True)
    for p in d.glob('*.json'):
        shutil.copy2(p, _ARCH / 'derived' / p.name)

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

ALL = ([m['id'] for m in G.get('MODELS', [])] +
       [m['id'] for m in G.get('NEW_MODELS', [])] +
       [m['id'] for m in G.get('EXTRA_MODELS', [])])
ALL = list(dict.fromkeys(ALL))

summary = {}
if 'ARM_IDS' in G:
    for mid in ALL:
        for arm in ('bench', 'post'):
            p = _RES / f'{mid}__rcm__{arm}.jsonl'
            if not p.exists(): continue
            want = set(G['ARM_IDS'][arm]); by = {}; stale = 0
            for line in open(p, encoding='utf-8'):
                if not line.strip(): continue
                r = json.loads(line)
                if r['cve'] in want: by[r['cve']] = r
                else: stale += 1
            n = len(by)
            got = sum(1 for r in by.values() if r.get('response'))
            ok  = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
            un  = sum(1 for r in by.values() if _parse(r['response']) is None)
            tin = sum((r.get('usage') or {}).get('prompt_tokens', 0) for r in by.values())
            tout= sum((r.get('usage') or {}).get('completion_tokens', 0) for r in by.values())
            summary[f'{mid}/{arm}'] = {
                'n': n, 'target': len(want), 'answered': got,
                'complete': n == len(want) and got == n,
                'acc': round(100*ok/n, 2) if n else None,
                'unparsed': un, 'stale_ignored': stale,
                'tokens_in': tin, 'tokens_out': tout}

manifest = {
    'archived_at': dt.datetime.now().isoformat(), 'stamp': _STAMP,
    'config': {k: str(G.get(k)) for k in
               ('PILOT_N','CUTOFF_BOUNDARY','SEED','TASKS','SNAP')},
    'models': G.get('MODELS', []), 'new_models': G.get('NEW_MODELS', []),
    'extra_models': G.get('EXTRA_MODELS', []),
    'arms': {'post': len(G.get('post_arm', [])),
             'bench': len(G.get('bench_arm', []))},
    'results': summary,
    'files': {'response_files': n_files, 'response_records': n_recs},
    'pickled_vars': sorted(saved), 'skipped_vars': skipped,
    'env': {'python': platform.python_version()},
}
try:
    import sklearn, numpy
    manifest['env'].update({'sklearn': sklearn.__version__,
                            'numpy': numpy.__version__})
except Exception:
    pass
(_ARCH / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

lines = []
for p in sorted(_ARCH.rglob('*')):
    if p.is_file() and p.name != 'CHECKSUMS.txt':
        lines.append(f'{hashlib.sha256(p.read_bytes()).hexdigest()[:16]}  '
                     f'{p.relative_to(_ARCH)}  {p.stat().st_size}')
(_ARCH / 'CHECKSUMS.txt').write_text('\n'.join(lines))

size = sum(p.stat().st_size for p in _ARCH.rglob('*') if p.is_file())
print(f'ARCHIVE -> {_ARCH}')
print(f'  pickled {len(saved)} vars | {n_files} response files '
      f'({n_recs} records) | {size/1e6:.1f} MB')
if skipped: print(f'  skipped: {skipped}')
print()
print(f'{"model/arm":20s}{"have":>7s}{"want":>7s}{"acc%":>8s}{"unp":>6s}{"status":>12s}')
incomplete = []
for k, v in summary.items():
    a = f'{v["acc"]:8.1f}' if v['acc'] is not None else f'{"-":>8s}'
    st = 'complete' if v['complete'] else 'PARTIAL'
    if not v['complete']: incomplete.append(k)
    print(f'{k:20s}{v["answered"]:7d}{v["target"]:7d}{a}{v["unparsed"]:6d}{st:>12s}')
print()
if incomplete:
    print(f'PARTIAL: {", ".join(incomplete)}')
    print('  -> safe. Re-run the model cell; it resumes from the JSONL.')
else:
    print('all arms complete')
print()
print('restore with:')
print(f"  import pickle; S = pickle.load(open('{_ARCH}/session.pkl','rb'))")

ARCHIVE -> /content/drive/MyDrive/ctibench-audit/archive/20260906_094229
  pickled 32 vars | 10 response files (15340 records) | 22.7 MB

model/arm              have   want    acc%   unp      status
gpt-4o/bench            976    997    75.5     0     PARTIAL
gpt-4o/post             967    997    75.5     0     PARTIAL
gpt-4.1/bench           989    997    74.0     0     PARTIAL
gpt-4.1/post            972    997    74.1     0     PARTIAL
gpt-5.5/bench           980    997    75.5     0     PARTIAL
gpt-5.5/post            953    997    80.1     0     PARTIAL
llama-4/bench           997    997    74.6     1    complete
llama-4/post            997    997    74.4     0    complete
claude/bench            997    997    74.3     0    complete
claude/post             997    997    76.4     0    complete

PARTIAL: gpt-4o/bench, gpt-4o/post, gpt-4.1/bench, gpt-4.1/post, gpt-5.5/bench, gpt-5.5/post
  -> safe. Re-run the model cell; it resumes from the JSONL.

restore with:
  import pickle; S = 

In [ ]:
OA_MODELS = [
    {'id': 'gpt-4o',  'model': 'gpt-4o-2024-08-06',  'cutoff': '2023-10-01'},
    {'id': 'gpt-4.1', 'model': 'gpt-4.1-2025-04-14', 'cutoff': '2024-06-01'},
    {'id': 'gpt-5.5', 'model': 'gpt-5.5-2026-04-23', 'cutoff': '2025-12-01'},
]
OA_WORKERS = 4
OA_TRIES   = 8

import json, re, time, random, threading, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

OA_BASE = 'https://api.openai.com/v1'
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')

assert 'post_arm' in globals() and 'bench_arm' in globals() and 'ARM_IDS' in globals(), \
    'Run the Stage 1 arm cells (sections 1-3) first.'
assert len(post_arm) == len(bench_arm), 'arms differ in size'
print(f'arms in memory: post={len(post_arm)}, bench={len(bench_arm)}')

try:
    OA_KEY
except NameError:
    try:
        from google.colab import userdata
        OA_KEY = (userdata.get('OPENAI_API_KEY') or '').strip()
    except Exception:
        OA_KEY = ''
    if not OA_KEY:
        from getpass import getpass
        OA_KEY = getpass('OPENAI_API_KEY: ').strip()
assert OA_KEY, 'no OpenAI key'

RCM_P = ('Analyze the following CVE description and map it to the appropriate CWE. '
         'Provide a brief justification for your choice. Ensure the last line of your '
         'response contains only the CWE ID.  CVE Description: ')

_hdr = {'Authorization': 'Bearer ' + OA_KEY, 'Content-Type': 'application/json'}
_style = {}
_stats = {'429': 0}

def _mk_body(m, prompt):
    st = _style.get(m['id'], {'tok': 'max_tokens', 'temp': True})
    b = {'model': m['model'], 'messages': [{'role': 'user', 'content': prompt}],
        st['tok']: 2500}
    if st['temp']: b['temperature'] = 0
    return b

def _fix(m, err):
    st = _style.get(m['id'], {'tok': 'max_tokens', 'temp': True}); ch = False
    if 'max_completion_tokens' in err and st['tok'] == 'max_tokens':
        st['tok'] = 'max_completion_tokens'; ch = True
    if 'temperature' in err and st['temp']:
        st['temp'] = False; ch = True
    if ch: _style[m['id']] = st; print(f"   [adapt] {m['id']} -> {st}")
    return ch

def oa_call(m, prompt, verbose=False):
    last = 'no attempt'
    for i in range(OA_TRIES):
        try:
            r = requests.post(OA_BASE + '/chat/completions',
                              json=_mk_body(m, prompt), headers=_hdr, timeout=180)
            if r.status_code == 200:
                j = r.json(); ch = j.get('choices') or []
                if not ch:
                    return None, {'error': f'200 no choices: {str(j)[:200]}'}
                return ch[0]['message']['content'], j.get('usage', {})
            if r.status_code == 400 and _fix(m, r.text):
                continue
            last = f'HTTP {r.status_code}: {r.text[:200]}'
            if r.status_code == 429:
                _stats['429'] += 1
                wait = float(r.headers.get('retry-after', 0) or 0) or min(60, 5*(i+1))
                if verbose: print(f'   429 -> waiting {wait:.0f}s')
                time.sleep(wait); continue
            if r.status_code in (500, 502, 503, 529):
                time.sleep(2 ** i + random.random()); continue
            return None, {'error': last}
        except requests.RequestException as ex:
            last = f'{type(ex).__name__}: {str(ex)[:150]}'
            time.sleep(2 ** i + random.random())
    return None, {'error': f'exhausted retries; last -> {last}'}

print('\n--- smoke test ---')
bad = []
for m in OA_MODELS:
    t, u = oa_call(m, RCM_P + post_arm[0]['description'] + ' ', verbose=True)
    print(('OK   ' if t else 'FAIL ') + f"{m['id']:10s} {str(t)[-50:]!r}")
    if not t: bad.append((m['id'], str(u.get('error'))[:160]))
assert not bad, f'smoke test failed: {bad}'

_lk = threading.Lock()
def refill(m, arm, items, dkey, gkey):
    out = _RES / f"{m['id']}__rcm__{arm}.jsonl"
    done = set()
    if out.exists():
        for line in open(out, encoding='utf-8'):
            if line.strip():
                r = json.loads(line)
                if r.get('response'):
                    done.add(r['cve'])
    todo = []
    for r in items:
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        if cve not in done: todo.append((cve, r))
    print(f"\n{m['id']} {arm}: {len(done)} done, {len(todo)} to go")
    if len(todo) > 300:
        print('   !! more than 300 missing -- arms may have redrawn. STOPPING.')
        return
    if not todo: return
    def work(pair):
        cve, r = pair
        t, u = oa_call(m, RCM_P + r[dkey] + ' ')
        return {'cve': cve, 'gt': r[gkey], 'response': t, 'usage': u,
                'model': m['id'], 'task': 'rcm', 'arm': arm}
    fails = 0
    with open(out, 'a', encoding='utf-8') as f, ThreadPoolExecutor(OA_WORKERS) as ex:
        for n, fut in enumerate(as_completed([ex.submit(work, p) for p in todo]), 1):
            rec = fut.result()
            if not rec['response']: fails += 1
            with _lk: f.write(json.dumps(rec) + '\n'); f.flush()
            if n % 25 == 0:
                print(f'   {n}/{len(todo)}  (429s: {_stats["429"]}, failed: {fails})')
    print(f'   finished: {len(todo)-fails} ok, {fails} failed')

for m in OA_MODELS:
    refill(m, 'post',  post_arm,  'description', 'GT_cwe')
    refill(m, 'bench', bench_arm, 'Description', 'GT')

_CWE = re.compile(r'CWE-\d+')
def _pp(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        mm = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if mm: return mm.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def score(mid, arm):
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    if not p.exists(): return None
    want = set(ARM_IDS[arm]); by = {}
    for line in open(p, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r['cve'] in want and r.get('response'): by[r['cve']] = r
    n = len(by)
    if not n: return None
    ok = sum(1 for r in by.values() if _pp(r['response']) == r['gt'])
    un = sum(1 for r in by.values() if _pp(r['response']) is None)
    return {'n': n, 'want': len(want), 'acc': 100*ok/n, 'unp': un}

ALL_IDS = ['gpt-4o','gpt-4.1','gpt-5.5','llama-4','claude']
print(f'\n{"model":12s}{"bench":>8s}{"post":>8s}{"n_b":>7s}{"n_p":>7s}{"status":>11s}')
bench_acc, post_acc, incomplete = {}, {}, []
for mid in ALL_IDS:
    b, p_ = score(mid, 'bench'), score(mid, 'post')
    if not b or not p_: continue
    bench_acc[mid], post_acc[mid] = b['acc'], p_['acc']
    full = (b['n'] == b['want'] and p_['n'] == p_['want'])
    if not full: incomplete.append(mid)
    print(f'{mid:12s}{b["acc"]:8.1f}{p_["acc"]:8.1f}{b["n"]:7d}{p_["n"]:7d}'
          f'{("complete" if full else "PARTIAL"):>11s}')

if bench_acc:
    bs, ps = list(bench_acc.values()), list(post_acc.values())
    print(f'\nbenchmark spread : {max(bs)-min(bs):.1f} pp  across {len(bs)} models')
    print(f'post-cutoff spread: {max(ps)-min(ps):.1f} pp')
print(f'429s this run: {_stats["429"]}')
if incomplete:
    print(f'\nstill PARTIAL: {incomplete} -- re-run this cell, it resumes.')
else:
    print('\nALL MODELS COMPLETE -- run the bootstrap cell next.')

arms in memory: post=997, bench=997

--- smoke test ---
OK   gpt-4o     'the description of a clickjacking issue.\n\nCWE-1021'
OK   gpt-4.1    ' attackers to overlay malicious content.\n\nCWE-1021'
   [adapt] gpt-5.5 -> {'tok': 'max_completion_tokens', 'temp': True}
   [adapt] gpt-5.5 -> {'tok': 'max_completion_tokens', 'temp': False}
OK   gpt-5.5    'ictions on rendered UI layers or frames.\n\nCWE-1021'

gpt-4o post: 1759 done, 0 to go

gpt-4o bench: 997 done, 0 to go

gpt-4.1 post: 1766 done, 0 to go

gpt-4.1 bench: 997 done, 0 to go

gpt-5.5 post: 1507 done, 9 to go
   finished: 8 ok, 1 failed

gpt-5.5 bench: 996 done, 1 to go
   finished: 1 ok, 0 failed

model          bench    post    n_b    n_p     status
gpt-4o          75.3    75.6    997    997   complete
gpt-4.1         73.8    74.4    997    997   complete
gpt-5.5         74.9    77.2    997    996    PARTIAL
llama-4         74.6    74.4    997    997   complete
claude          74.3    76.4    997    997   complete

benchmar

In [ ]:
import json, re
missing = {}
for arm in ('post','bench'):
    p = _RES / f'gpt-5.5__rcm__{arm}.jsonl'
    have = {json.loads(l)['cve'] for l in open(p, encoding='utf-8')
            if l.strip() and json.loads(l).get('response')}
    items = post_arm if arm == 'post' else bench_arm
    miss = []
    for r in items:
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        if cve not in have:
            miss.append((cve, r.get('description') or r.get('Description')))
    missing[arm] = miss
    print(f'{arm}: {len(miss)} missing')

lens = [len(d) for m in missing.values() for _, d in m]
allen = [len(r.get('description','')) for r in post_arm]
print(f'\nfailing desc length: mean {sum(lens)/len(lens):.0f}, max {max(lens)}')
print(f'all items:           mean {sum(allen)/len(allen):.0f}, max {max(allen)}')

cve, desc = missing['post'][0]
print(f'\nprobing {cve} (len {len(desc)}):')
t, u = oa_call(OA_MODELS[2], RCM_P + desc + ' ', verbose=True)
print('  got text?', t is not None)
print('  usage/error:', str(u)[:300])

post: 9 missing
bench: 1 missing

failing desc length: mean 506, max 958
all items:           mean 403, max 3998

probing CVE-2026-19848 (len 317):
  got text? True
  usage/error: {'prompt_tokens': 110, 'completion_tokens': 800, 'total_tokens': 910, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 800, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}


In [ ]:
MODELS = [
 {'id':'gpt-4o','model':'gpt-4o-2024-08-06','cutoff':'2023-10-01',
  'cutoff_src':'https://developers.openai.com/api/docs/models/gpt-4o'},
 {'id':'gpt-4.1','model':'gpt-4.1-2025-04-14','cutoff':'2024-06-01',
  'cutoff_src':'https://developers.openai.com/api/docs/models/gpt-4.1'},
 {'id':'gpt-5.5','model':'gpt-5.5-2026-04-23','cutoff':'2025-12-01',
  'cutoff_src':'https://developers.openai.com/api/docs/models/gpt-5.5'},
]
NEW_MODELS = [
 {'id':'llama-4','model':'meta-llama/llama-4-maverick','cutoff':'2024-08-31',
  'cutoff_src':'https://www.llama.com/docs/model-cards-and-prompt-formats/llama4/'},
 {'id':'claude','model':'anthropic/claude-sonnet-4.5','cutoff':'2025-07-01',
  'cutoff_src':'https://docs.claude.com/en/docs/about-claude/models/overview'},
]
print('registered:', [m['id'] for m in MODELS + NEW_MODELS])

registered: ['gpt-4o', 'gpt-4.1', 'gpt-5.5', 'llama-4', 'claude']


In [ ]:
SAVE_BIG_POOLS = False

import os, json, pickle, shutil, hashlib, platform, re
import datetime as dt
from pathlib import Path

_PROJ  = Path('/content/drive/MyDrive/ctibench-audit')
_RES   = _PROJ / 'results'
_STAMP = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
_ARCH  = _PROJ / 'archive' / _STAMP
(_ARCH / 'results').mkdir(parents=True, exist_ok=True)
G = globals()

_SMALL = ['PILOT_N','TASKS','MAX_WORKERS','CUTOFF_BOUNDARY','SEED','SNAP',
          'MODELS','NEW_MODELS','EXTRA_MODELS','OA_MODELS','ACTIVE','PROVIDERS',
          'PARAM_STYLE','_STYLE','_style','CUTOFFS','BENCH_HIST','BENCH_CWE_DIST',
          'COMMON','EXCLUDED','target','post_arm','bench_arm','ARM_IDS',
          'rcm_bench','vsp_bench','acc_b','acc_p','BENCH_LABELS',
          'DUP_THRESHOLDS','audit','RCM_PROMPT','VSP_HEAD','RCM','RCM_P','maj',
          'spans','boot','V','summary','bench_acc','post_acc']
_BIG = ['clean','post','post_pool_all','clean_new','clean_old','rows','pool']

saved, skipped = {}, []
for name in _SMALL + (_BIG if SAVE_BIG_POOLS else []):
    if name not in G:
        continue
    try:
        obj = G[name]
        if name == 'PROVIDERS':
            obj = {k: {kk: vv for kk, vv in v.items() if 'key' not in kk.lower()}
                   for k, v in obj.items()}
        if isinstance(obj, Path):
            obj = str(obj)
        pickle.dumps(obj)
        saved[name] = obj
    except Exception as e:
        skipped.append(f'{name}: {type(e).__name__}')

with open(_ARCH / 'session.pkl', 'wb') as f:
    pickle.dump(saved, f, protocol=4)

n_files = n_recs = 0
if _RES.exists():
    for p in sorted(_RES.glob('*.jsonl')):
        shutil.copy2(p, _ARCH / 'results' / p.name)
        n_files += 1
        n_recs += sum(1 for _ in open(p, encoding='utf-8'))
d = _PROJ / 'derived'
if d.exists():
    (_ARCH / 'derived').mkdir(exist_ok=True)
    for p in d.glob('*.json'):
        shutil.copy2(p, _ARCH / 'derived' / p.name)

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

ALL = ([m['id'] for m in G.get('MODELS', [])] +
       [m['id'] for m in G.get('NEW_MODELS', [])] +
       [m['id'] for m in G.get('EXTRA_MODELS', [])])
ALL = list(dict.fromkeys(ALL))

summary = {}
if 'ARM_IDS' in G:
    for mid in ALL:
        for arm in ('bench', 'post'):
            p = _RES / f'{mid}__rcm__{arm}.jsonl'
            if not p.exists(): continue
            want = set(G['ARM_IDS'][arm]); by = {}; stale = 0
            for line in open(p, encoding='utf-8'):
                if not line.strip(): continue
                r = json.loads(line)
                if r['cve'] in want:
                    if r.get('response'): by[r['cve']] = r
                else: stale += 1
            n = len(by)
            ok  = sum(1 for r in by.values() if _parse(r['response']) == r['gt'])
            un  = sum(1 for r in by.values() if _parse(r['response']) is None)
            tin = sum((r.get('usage') or {}).get('prompt_tokens', 0) for r in by.values())
            tout= sum((r.get('usage') or {}).get('completion_tokens', 0) for r in by.values())
            summary[f'{mid}/{arm}'] = {
                'n': n, 'target': len(want), 'answered': n,
                'complete': n == len(want),
                'acc': round(100*ok/n, 2) if n else None,
                'unparsed': un, 'stale_ignored': stale,
                'tokens_in': tin, 'tokens_out': tout}

manifest = {
    'archived_at': dt.datetime.now().isoformat(), 'stamp': _STAMP,
    'config': {k: str(G.get(k)) for k in
               ('PILOT_N','CUTOFF_BOUNDARY','SEED','TASKS','SNAP')},
    'models': G.get('MODELS', []), 'new_models': G.get('NEW_MODELS', []),
    'extra_models': G.get('EXTRA_MODELS', []),
    'arms': {'post': len(G.get('post_arm', [])),
             'bench': len(G.get('bench_arm', []))},
    'results': summary,
    'files': {'response_files': n_files, 'response_records': n_recs},
    'pickled_vars': sorted(saved), 'skipped_vars': skipped,
    'env': {'python': platform.python_version()},
}
try:
    import sklearn, numpy
    manifest['env'].update({'sklearn': sklearn.__version__,
                            'numpy': numpy.__version__})
except Exception:
    pass
(_ARCH / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))

lines = []
for p in sorted(_ARCH.rglob('*')):
    if p.is_file() and p.name != 'CHECKSUMS.txt':
        lines.append(f'{hashlib.sha256(p.read_bytes()).hexdigest()[:16]}  '
                     f'{p.relative_to(_ARCH)}  {p.stat().st_size}')
(_ARCH / 'CHECKSUMS.txt').write_text('\n'.join(lines))

size = sum(p.stat().st_size for p in _ARCH.rglob('*') if p.is_file())
print(f'ARCHIVE -> {_ARCH}')
print(f'  pickled {len(saved)} vars | {n_files} response files '
      f'({n_recs} records) | {size/1e6:.1f} MB')
if skipped: print(f'  skipped: {skipped}')
print()
print(f'{"model/arm":20s}{"have":>7s}{"want":>7s}{"acc%":>8s}{"unp":>6s}{"status":>12s}')
incomplete = []
for k, v in summary.items():
    a = f'{v["acc"]:8.1f}' if v['acc'] is not None else f'{"-":>8s}'
    st = 'complete' if v['complete'] else 'PARTIAL'
    if not v['complete']: incomplete.append(k)
    print(f'{k:20s}{v["answered"]:7d}{v["target"]:7d}{a}{v["unparsed"]:6d}{st:>12s}')
print()
if incomplete:
    print(f'PARTIAL: {", ".join(incomplete)}')
else:
    print('all arms complete')
print()
print('restore with:')
print(f"  import pickle; S = pickle.load(open('{_ARCH}/session.pkl','rb'))")

ARCHIVE -> /content/drive/MyDrive/ctibench-audit/archive/20260906_100223
  pickled 37 vars | 10 response files (15513 records) | 22.8 MB

model/arm              have   want    acc%   unp      status
gpt-4o/bench            997    997    75.3     0    complete
gpt-4o/post             997    997    75.6     0    complete
gpt-4.1/bench           997    997    73.8     0    complete
gpt-4.1/post            997    997    74.4     0    complete
gpt-5.5/bench           997    997    74.9     0    complete
gpt-5.5/post            996    997    77.2     0     PARTIAL
llama-4/bench           997    997    74.6     1    complete
llama-4/post            997    997    74.4     0    complete
claude/bench            997    997    74.3     0    complete
claude/post             997    997    76.4     0    complete

PARTIAL: gpt-5.5/post

restore with:
  import pickle; S = pickle.load(open('/content/drive/MyDrive/ctibench-audit/archive/20260906_100223/session.pkl','rb'))


In [ ]:
import json, re, random
import numpy as np
from itertools import combinations
from math import comb
from pathlib import Path

N_BOOT = 10000
random.seed(20261005); np.random.seed(20261005)
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
IDS = ['gpt-4o','gpt-4.1','gpt-5.5','llama-4','claude']

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def vec(mid, arm):
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    d = {}
    for line in open(p, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r.get('response'):
                d[r['cve']] = int(_parse(r['response']) == r['gt'])
    return d

# COMMON ITEMS ONLY: every model must have answered, so pairing is valid
COMMON = {}
for arm in ('bench','post'):
    sets = [set(vec(m, arm)) for m in IDS]
    COMMON[arm] = sorted(set(ARM_IDS[arm]) & set.intersection(*sets))
    print(f'{arm}: {len(COMMON[arm])} items answered by all 5 models')

V = {}
for m in IDS:
    for arm in ('bench','post'):
        d = vec(m, arm)
        V[(m,arm)] = np.array([d[c] for c in COMMON[arm]], float)

nb, npp = len(COMMON['bench']), len(COMMON['post'])
idx = {'bench': np.random.randint(0,nb,size=(N_BOOT,nb)),
       'post':  np.random.randint(0,npp,size=(N_BOOT,npp))}
boot = {k: V[k][idx[k[1]]].mean(axis=1)*100 for k in V}
def ci(a): return np.percentile(a,2.5), np.percentile(a,97.5)

print(f'\n{"model":10s}{"bench":>18s}{"post":>18s}')
for m in IDS:
    b, p_ = V[(m,'bench')].mean()*100, V[(m,'post')].mean()*100
    lb, hb = ci(boot[(m,'bench')]); lp, hp = ci(boot[(m,'post')])
    print(f'{m:10s}{b:7.1f} [{lb:.1f},{hb:.1f}]{p_:8.1f} [{lp:.1f},{hp:.1f}]')

print('\nPAIRED comparisons (same items, exact McNemar):')
print(f'{"arm":7s}{"pair":26s}{"diff":>7s}{"95% CI":>17s}{"p":>10s}')
sig = {'bench':0,'post':0}
for arm in ('bench','post'):
    for a,b in combinations(IDS,2):
        va, vb = V[(a,arm)], V[(b,arm)]
        bd = (va-vb)[idx[arm]].mean(axis=1)*100
        lo,hi = ci(bd)
        n01 = int(((va==0)&(vb==1)).sum()); n10 = int(((va==1)&(vb==0)).sum())
        n = n01+n10
        p = 1.0 if n==0 else min(1.0, 2*sum(comb(n,k) for k in range(0,min(n01,n10)+1))/(2**n))
        star = '  *' if p < 0.05 else ''
        if p < 0.05: sig[arm]+= 1
        print(f'{arm:7s}{a+" vs "+b:26s}{(va-vb).mean()*100:7.1f}   [{lo:5.1f},{hi:5.1f}]{p:10.4f}{star}')

print(f'\nsignificant pairs: benchmark {sig["bench"]}/10, post-cutoff {sig["post"]}/10')

print('\nEquivalence bounds on the benchmark (what it can rule out):')
worst = 0
for a,b in combinations(IDS,2):
    bd = (V[(a,'bench')]-V[(b,'bench')])[idx['bench']].mean(axis=1)*100
    lo,hi = ci(bd); w = max(abs(lo),abs(hi)); worst = max(worst,w)
    print(f'  {a:9s} vs {b:9s}: |diff| < {w:.1f} pp')
print(f'  -> the benchmark cannot resolve differences above {worst:.1f} pp')

print('\nArm gaps (bench - post) and DiD vs unexposed control gpt-4o:')
gc = boot[('gpt-4o','bench')] - boot[('gpt-4o','post')]
for m in IDS:
    g = boot[(m,'bench')] - boot[(m,'post')]
    pt = V[(m,'bench')].mean()*100 - V[(m,'post')].mean()*100
    lo,hi = ci(g)
    extra = ''
    if m != 'gpt-4o':
        dd = g - gc; dlo,dhi = ci(dd)
        extra = f'   DiD {dd.mean():+5.1f} [{dlo:+5.1f},{dhi:+5.1f}]'
    print(f'  {m:10s}{pt:+6.1f} [{lo:+5.1f},{hi:+5.1f}]{extra}')

sp = {}
for arm in ('bench','post'):
    st = np.vstack([boot[(m,arm)] for m in IDS])
    sp[arm] = st.max(axis=0)-st.min(axis=0)
    pt = max(V[(m,arm)].mean() for m in IDS)*100 - min(V[(m,arm)].mean() for m in IDS)*100
    lo,hi = ci(sp[arm])
    print(f'\n{arm} spread: {pt:.1f} pp  95% CI [{lo:.1f},{hi:.1f}]')
dd = sp['post']-sp['bench']; lo,hi = ci(dd)
print(f'post - bench spread: {dd.mean():+.1f}  95% CI [{lo:+.1f},{hi:+.1f}]  '
      f'P(>0) = {(dd>0).mean():.3f}')

bench: 997 items answered by all 5 models
post: 996 items answered by all 5 models

model                  bench              post
gpt-4o       75.3 [72.6,77.9]    75.7 [73.0,78.3]
gpt-4.1      73.8 [71.0,76.5]    74.5 [71.9,77.2]
gpt-5.5      74.9 [72.2,77.6]    77.2 [74.6,79.8]
llama-4      74.6 [71.8,77.3]    74.5 [71.8,77.2]
claude       74.3 [71.5,77.0]    76.5 [73.9,79.1]

PAIRED comparisons (same items, exact McNemar):
arm    pair                         diff           95% CI         p
bench  gpt-4o vs gpt-4.1             1.5   [  0.1,  2.9]    0.0534
bench  gpt-4o vs gpt-5.5             0.4   [ -1.5,  2.2]    0.7465
bench  gpt-4o vs llama-4             0.7   [ -0.9,  2.3]    0.4638
bench  gpt-4o vs claude              1.0   [ -0.9,  2.9]    0.3533
bench  gpt-4.1 vs gpt-5.5           -1.1   [ -2.9,  0.7]    0.2723
bench  gpt-4.1 vs llama-4           -0.8   [ -2.5,  0.9]    0.4160
bench  gpt-4.1 vs claude            -0.5   [ -2.4,  1.4]    0.6817
bench  gpt-5.5 vs llama-4        

In [ ]:
import json, re, numpy as np
from itertools import combinations
from math import comb
from pathlib import Path
from scipy import stats

_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
IDS = ['gpt-4o','gpt-4.1','gpt-5.5','llama-4','claude']
SESOI = 5.0   # smallest difference a benchmark should resolve, in pp. JUSTIFY THIS.

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def vec(mid, arm):
    d = {}
    for line in open(_RES/f'{mid}__rcm__{arm}.jsonl', encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r.get('response'):
                d[r['cve']] = int(_parse(r['response']) == r['gt'])
    return d

COMMON, V = {}, {}
for arm in ('bench','post'):
    sets = [set(vec(m,arm)) for m in IDS]
    COMMON[arm] = sorted(set(ARM_IDS[arm]) & set.intersection(*sets))
    for m in IDS:
        d = vec(m,arm)
        V[(m,arm)] = np.array([d[c] for c in COMMON[arm]], float)
    print(f'{arm}: n={len(COMMON[arm])}')

# ---------- PRIMARY: Cochran's Q, one omnibus test per arm ----------
print('\n=== Cochran Q (omnibus, k=5 models, paired items) ===')
for arm in ('bench','post'):
    X = np.vstack([V[(m,arm)] for m in IDS]).T      # items x models
    n, k = X.shape
    Gj = X.sum(axis=0); Li = X.sum(axis=1); N = X.sum()
    Q = (k-1)*(k*(Gj**2).sum() - N**2) / (k*N - (Li**2).sum())
    p = 1 - stats.chi2.cdf(Q, k-1)
    print(f'{arm:6s} Q={Q:7.3f}  df={k-1}  p={p:.4f}   '
          f'{"models DIFFER" if p<0.05 else "no detectable difference"}')

# ---------- SECONDARY: pairwise with Holm ----------
print('\n=== Pairwise, Holm-adjusted (exploratory) ===')
for arm in ('bench','post'):
    res = []
    for a,b in combinations(IDS,2):
        va, vb = V[(a,arm)], V[(b,arm)]
        n01 = int(((va==0)&(vb==1)).sum()); n10 = int(((va==1)&(vb==0)).sum()); n = n01+n10
        p = 1.0 if n==0 else min(1.0, 2*sum(comb(n,i) for i in range(0,min(n01,n10)+1))/(2**n))
        res.append((f'{a} vs {b}', (va-vb).mean()*100, p))
    res.sort(key=lambda r: r[2])
    run = 0.0
    print(f'--- {arm} ---')
    for i,(name,d,p) in enumerate(res):
        run = max(run, min(1.0, p*(len(res)-i)))
        flag = ' *' if run < 0.05 else ''
        if i < 3 or run < 0.05:
            print(f'  {name:24s} {d:+6.2f} pp  raw={p:.4f}  Holm={run:.3f}{flag}')
    print(f'  significant after Holm: {sum(1 for _,_,q in res if q<0.05 and False)}'
          f'  (see flags above)')

# ---------- TOST equivalence, pre-specified margin ----------
print(f'\n=== TOST equivalence, margin = {SESOI} pp (benchmark arm) ===')
maxp = 0
for a,b in combinations(IDS,2):
    va, vb = V[(a,'bench')], V[(b,'bench')]
    d = va - vb
    diff = d.mean()*100
    se = d.std(ddof=1)/np.sqrt(len(d))*100
    t_lo = (diff + SESOI)/se; t_hi = (diff - SESOI)/se
    df = len(d)-1
    p_lo = 1 - stats.t.cdf(t_lo, df); p_hi = stats.t.cdf(t_hi, df)
    p_tost = max(p_lo, p_hi); maxp = max(maxp, p_tost)
    print(f'  {a:9s} vs {b:9s} diff={diff:+5.2f} pp  TOST p={p_tost:.4f}'
          f'  {"EQUIVALENT" if p_tost<0.05 else "not shown"}')
print(f'\nAll pairs equivalent at {SESOI} pp? {"YES" if maxp<0.05 else "NO"}  (max p={maxp:.4f})')

bench: n=997
post: n=996

=== Cochran Q (omnibus, k=5 models, paired items) ===
bench  Q=  3.278  df=4  p=0.5124   no detectable difference
post   Q= 10.449  df=4  p=0.0335   models DIFFER

=== Pairwise, Holm-adjusted (exploratory) ===
--- bench ---
  gpt-4o vs gpt-4.1         +1.50 pp  raw=0.0534  Holm=0.534
  gpt-4.1 vs gpt-5.5        -1.10 pp  raw=0.2723  Holm=1.000
  gpt-4o vs claude          +1.00 pp  raw=0.3533  Holm=1.000
  significant after Holm: 0  (see flags above)
--- post ---
  gpt-4.1 vs gpt-5.5        -2.71 pp  raw=0.0132  Holm=0.132
  gpt-5.5 vs llama-4        +2.71 pp  raw=0.0141  Holm=0.132
  gpt-4.1 vs claude         -2.01 pp  raw=0.0798  Holm=0.639
  significant after Holm: 0  (see flags above)

=== TOST equivalence, margin = 5.0 pp (benchmark arm) ===
  gpt-4o    vs gpt-4.1   diff=+1.50 pp  TOST p=0.0000  EQUIVALENT
  gpt-4o    vs gpt-5.5   diff=+0.40 pp  TOST p=0.0000  EQUIVALENT
  gpt-4o    vs llama-4   diff=+0.70 pp  TOST p=0.0000  EQUIVALENT
  gpt-4o    vs claud

In [ ]:
try:
    import statsmodels
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])

import json, re, numpy as np, pandas as pd
from itertools import combinations
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable

_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
IDS  = ['gpt-4o','gpt-4.1','gpt-5.5','llama-4','claude']
REF  = 'gpt-4o'

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def vec(mid, arm):
    d = {}
    for line in open(_RES/f'{mid}__rcm__{arm}.jsonl', encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r.get('response'):
                d[r['cve']] = (int(_parse(r['response']) == r['gt']), r['gt'])
    return d

rows, COMMON, V = [], {}, {}
for arm in ('bench','post'):
    sets = [set(vec(m,arm)) for m in IDS]
    COMMON[arm] = sorted(set(ARM_IDS[arm]) & set.intersection(*sets))
    got = {m: vec(m,arm) for m in IDS}
    for c in COMMON[arm]:
        for m in IDS:
            ok, gt = got[m][c]
            rows.append({'correct': ok, 'model': m, 'arm': arm,
                         'item': f'{arm}:{c}', 'cwe': gt})
    for m in IDS:
        V[(m,arm)] = np.array([got[m][c][0] for c in COMMON[arm]], float)
    print(f'{arm}: n={len(COMMON[arm])}')

df = pd.DataFrame(rows)
df['model'] = pd.Categorical(df['model'], categories=[REF] + [m for m in IDS if m != REF])
df['arm']   = pd.Categorical(df['arm'],   categories=['bench','post'])
print(f'observations: {len(df)}  ({df.item.nunique()} items x {len(IDS)} models)')

print('\n=== GEE: correct ~ model * arm, clustered by item ===')
full = smf.gee('correct ~ C(model) * C(arm)', groups='item', data=df,
               family=Binomial(), cov_struct=Exchangeable()).fit()

inter = [c for c in full.params.index if ':' in c]
R = np.zeros((len(inter), len(full.params)))
for i, name in enumerate(inter):
    R[i, list(full.params.index).index(name)] = 1.0
wt = full.wald_test(R, scalar=False)
chi2 = float(np.squeeze(wt.statistic)); pval = float(np.squeeze(wt.pvalue))
print(f'\nJOINT INTERACTION  chi2={chi2:.3f}  df={len(inter)}  p={pval:.4f}')
print('  ->', 'model ranking DIFFERS between arms (dissociation supported)'
      if pval < 0.05 else
      'NO detectable interaction; report the arm contrast as descriptive only')

print('\nInteraction coefficients (log-odds, vs. reference model and bench arm):')
ci = full.conf_int()
for name in inter:
    lo, hi = ci.loc[name]
    print(f'  {name:38s} {full.params[name]:+.3f}  [{lo:+.3f},{hi:+.3f}]'
          f'  p={full.pvalues[name]:.3f}')

print('\n=== TOST sensitivity (paired item-level differences) ===')
for margin in (3.0, 4.0, 5.0):
    print(f'\n--- margin = {margin} pp ---')
    for arm in ('bench','post'):
        worst_p, widest = 0.0, ''
        for a,b in combinations(IDS,2):
            d = V[(a,arm)] - V[(b,arm)]
            diff = d.mean()*100
            se = d.std(ddof=1)/np.sqrt(len(d))*100
            dof = len(d)-1
            p_t = max(1-stats.t.cdf((diff+margin)/se, dof),
                      stats.t.cdf((diff-margin)/se, dof))
            lo90 = diff - stats.t.ppf(0.95, dof)*se
            hi90 = diff + stats.t.ppf(0.95, dof)*se
            if p_t > worst_p:
                worst_p, widest = p_t, f'{a} vs {b}: {diff:+.2f} [90% CI {lo90:+.2f},{hi90:+.2f}]'
        verdict = 'ALL equivalent' if worst_p < 0.05 else 'NOT all equivalent'
        print(f'  {arm:6s} {verdict:18s} worst p={worst_p:.4f}   {widest}')

bench: n=997
post: n=996
observations: 9965  (1993 items x 5 models)

=== GEE: correct ~ model * arm, clustered by item ===

JOINT INTERACTION  chi2=3.853  df=4  p=0.4263
  -> NO detectable interaction; report the arm contrast as descriptive only

Interaction coefficients (log-odds, vs. reference model and bench arm):
  C(model)[T.gpt-4.1]:C(arm)[T.post]     +0.015  [-0.112,+0.142]  p=0.818
  C(model)[T.gpt-5.5]:C(arm)[T.post]     +0.105  [-0.047,+0.258]  p=0.176
  C(model)[T.llama-4]:C(arm)[T.post]     -0.027  [-0.161,+0.107]  p=0.693
  C(model)[T.claude]:C(arm)[T.post]      +0.097  [-0.062,+0.257]  p=0.231

=== TOST sensitivity (paired item-level differences) ===

--- margin = 3.0 pp ---
  bench  ALL equivalent     worst p=0.0202   gpt-4o vs gpt-4.1: +1.50 [90% CI +0.30,+2.70]
  post   NOT all equivalent worst p=0.3930   gpt-5.5 vs llama-4: +2.71 [90% CI +0.96,+4.46]

--- margin = 4.0 pp ---
  bench  ALL equivalent     worst p=0.0011   gpt-4o vs claude: +1.00 [90% CI -0.60,+2.60]
  p

In [ ]:
# ============================================================================
# CLUSTER-ROBUST EQUIVALENCE SENSITIVITY
#
# Clustering rule (does NOT use model outcomes):
#   TF-IDF word 1-2 grams over the EVALUATED items' descriptions in each arm;
#   cosine similarity; undirected graph with an edge wherever cos >= tau;
#   connected components = clusters. Singletons are clusters of size 1.
#
# Bootstrap: resample CLUSTERS with replacement (B draws). All five models'
#   outcomes for an item travel together. Accuracy differences are
#   ITEM-WEIGHTED over the resampled item multiset (not means of cluster means).
#
# Equivalence: declared iff the 90% cluster-bootstrap CI for the paired
#   difference lies entirely inside (-margin, +margin). No bootstrap p-values
#   are invented; the global "all pairs" claim is an intersection-union test,
#   so it holds iff every pair's CI is inside the margin.
# ============================================================================

B_BOOT     = 5000
THRESHOLDS = (0.9, 0.8)
MARGINS    = (3.0, 4.0, 5.0)
SEED       = 20261005

import json, re, numpy as np
from itertools import combinations
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
IDS  = ['gpt-4o','gpt-4.1','gpt-5.5','llama-4','claude']

_CWE = re.compile(r'CWE-\d+')
def _parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def outcomes(mid, arm):
    d = {}
    for line in open(_RES/f'{mid}__rcm__{arm}.jsonl', encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            if r.get('response'):
                d[r['cve']] = int(_parse(r['response']) == r['gt'])
    return d

# ---- evaluated item ids ONLY: intersection across all five models ----
ITEMS, TEXT, V = {}, {}, {}
for arm in ('bench','post'):
    got = {m: outcomes(m, arm) for m in IDS}
    ids = sorted(set(ARM_IDS[arm]).intersection(*[set(g) for g in got.values()]))
    ITEMS[arm] = ids
    src = post_arm if arm == 'post' else bench_arm
    key = 'description' if arm == 'post' else 'Description'
    idx = {}
    for r in src:
        cve = r.get('cve') or re.search(r'CVE-\d{4}-\d+', r['URL']).group(0)
        idx[cve] = r[key]
    TEXT[arm] = [idx[c] for c in ids]
    for m in IDS:
        V[(m,arm)] = np.array([got[m][c] for c in ids], float)
    print(f'{arm}: {len(ids)} evaluated items')

def components(texts, tau, block=500):
    """Connected components at cosine >= tau. Blocked to bound memory."""
    v = TfidfVectorizer(ngram_range=(1,2), min_df=1, sublinear_tf=True)
    X = v.fit_transform(texts)
    n = X.shape[0]
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    for i in range(0, n, block):
        S = cosine_similarity(X[i:i+block], X)
        ii, jj = np.where(S >= tau)
        for a, b in zip(ii + i, jj):
            if a < b: union(int(a), int(b))
    lab = np.array([find(i) for i in range(n)])
    _, comp = np.unique(lab, return_inverse=True)
    return comp

print(f'\ncluster bootstrap: B={B_BOOT}, seed={SEED}')
for tau in THRESHOLDS:
    print(f'\n{"="*66}\nCLUSTERING AT COSINE >= {tau}\n{"="*66}')
    draws, ncl = {}, {}
    for arm in ('bench','post'):
        comp = components(TEXT[arm], tau)
        k = comp.max() + 1
        ncl[arm] = k
        members = [np.where(comp == c)[0] for c in range(k)]
        sizes = np.array([len(m) for m in members])
        print(f'  {arm}: {len(comp)} items -> {k} clusters '
              f'(largest {sizes.max()}, singletons {(sizes==1).sum()})')
        # resample clusters with replacement; concatenate member items
        idxs = []
        for _ in range(B_BOOT):
            pick = rng.integers(0, k, size=k)
            idxs.append(np.concatenate([members[c] for c in pick]))
        draws[arm] = idxs

    # ---- measure the dependence effect: cluster CI width / item CI width ----
    print('\n  design effect (cluster CI width / item CI width, 90%):')
    for arm in ('bench','post'):
        nI = len(ITEMS[arm])
        item_draws = [rng.integers(0, nI, size=nI) for _ in range(1000)]
        ratios = []
        for a, b in combinations(IDS, 2):
            d = V[(a,arm)] - V[(b,arm)]
            cb = np.array([d[ix].mean() for ix in draws[arm][:1000]]) * 100
            ib = np.array([d[ix].mean() for ix in item_draws]) * 100
            wc = np.diff(np.percentile(cb, [5, 95]))[0]
            wi = np.diff(np.percentile(ib, [5, 95]))[0]
            ratios.append(wc / wi if wi > 0 else np.nan)
        r = np.array(ratios)
        print(f'    {arm:6s} median {np.nanmedian(r):.2f}x, range '
              f'{np.nanmin(r):.2f}-{np.nanmax(r):.2f}  '
              f'({"widens" if np.nanmedian(r) > 1.02 else "no material change"})')

    for margin in MARGINS:
        print(f'\n  --- margin = {margin} pp ---')
        for arm in ('bench','post'):
            worst, worst_pair, all_in = None, '', True
            for a, b in combinations(IDS, 2):
                d = V[(a,arm)] - V[(b,arm)]
                # item-weighted mean over each resampled item multiset
                bs = np.array([d[ix].mean() for ix in draws[arm]]) * 100
                lo, hi = np.percentile(bs, [5, 95])
                inside = (lo > -margin) and (hi < margin)
                all_in &= inside
                slack = max(abs(lo), abs(hi))
                if worst is None or slack > worst:
                    worst, worst_pair = slack, (
                        f'{a} vs {b}: {d.mean()*100:+.2f} '
                        f'[90% CI {lo:+.2f},{hi:+.2f}]{"" if inside else "  <-- outside"}')
            verdict = 'ALL pairs equivalent' if all_in else 'equivalence NOT established'
            print(f'    {arm:6s} {verdict:28s} widest: {worst_pair}')

print('\nInterpretation notes:')
print(' - "equivalence NOT established" does not mean the pair is non-equivalent.')
print(' - The global all-pairs claim is an intersection-union test: it holds')
print('   iff every pairwise CI lies inside the margin, with no multiplicity')
print('   adjustment required.')
print(' - Margins were fixed after inspecting item-level results; report as')
print('   exploratory sensitivity analysis, not a pre-specified test.')

bench: 997 evaluated items
post: 996 evaluated items

cluster bootstrap: B=5000, seed=20261005

CLUSTERING AT COSINE >= 0.9
  bench: 997 items -> 954 clusters (largest 11, singletons 933)
  post: 996 items -> 971 clusters (largest 5, singletons 953)

  design effect (cluster CI width / item CI width, 90%):
    bench  median 1.00x, range 0.95-1.06  (no material change)
    post   median 1.02x, range 1.00-1.10  (widens)

  --- margin = 3.0 pp ---
    bench  ALL pairs equivalent         widest: gpt-4.1 vs gpt-5.5: -1.10 [90% CI -2.72,+0.40]
    post   equivalence NOT established  widest: gpt-5.5 vs llama-4: +2.71 [90% CI +0.90,+4.50]  <-- outside

  --- margin = 4.0 pp ---
    bench  ALL pairs equivalent         widest: gpt-4.1 vs gpt-5.5: -1.10 [90% CI -2.72,+0.40]
    post   equivalence NOT established  widest: gpt-5.5 vs llama-4: +2.71 [90% CI +0.90,+4.50]  <-- outside

  --- margin = 5.0 pp ---
    bench  ALL pairs equivalent         widest: gpt-4.1 vs gpt-5.5: -1.10 [90% CI -2.72,+0.

In [2]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [3]:
# ============================================================================
# MACRO METRICS AND PER-CLASS BREAKDOWN  -- fully self-contained.
# Reads only files on Drive: results/*.jsonl and derived/arms_*.json
# No dependency on variables from earlier cells. Safe after a restart.
# ============================================================================

import json, re, glob
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, warnings
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score
warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')

MIN_SUPPORT = 10          # threshold for the "well-supported classes" cut
_PROJ = Path('/content/drive/MyDrive/ctibench-audit')
_RES, _DER = _PROJ / 'results', _PROJ / 'derived'

# ---- 1. recover arm definitions from disk ----------------------------------
arm_files = sorted(_DER.glob('arms_*.json'))
assert arm_files, f'no arms_*.json in {_DER}; cannot recover the evaluated sample'
ARMS_PATH = arm_files[-1]
_saved = json.loads(ARMS_PATH.read_text())
ARM_IDS = {'bench': list(_saved['bench']), 'post': list(_saved['post'])}
print(f'arms from {ARMS_PATH.name}: bench={len(ARM_IDS["bench"])}, '
      f'post={len(ARM_IDS["post"])}')

# ---- 2. discover which models have results ---------------------------------
MODEL_IDS = sorted({Path(f).name.split('__')[0] for f in glob.glob(str(_RES/'*__rcm__*.jsonl'))})
print(f'models found: {MODEL_IDS}')
assert MODEL_IDS, 'no result files found'

# ---- 3. parse (identical rule to the paper) --------------------------------
_CWE = re.compile(r'CWE-\d+')
def parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def load(mid, arm):
    """Returns {cve: (pred, gold)} for items in the current arm with a response."""
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    if not p.exists(): return {}
    want = set(ARM_IDS[arm]); out = {}
    for line in open(p, encoding='utf-8'):
        if not line.strip(): continue
        r = json.loads(line)
        if r['cve'] in want and r.get('response'):
            out[r['cve']] = (parse(r['response']), r['gt'].strip())
    return out

# common items: answered by every model, so all metrics are on the same set
COMMON = {}
for arm in ('bench','post'):
    sets = [set(load(m, arm)) for m in MODEL_IDS]
    sets = [s for s in sets if s]
    COMMON[arm] = sorted(set(ARM_IDS[arm]).intersection(*sets))
    print(f'{arm}: {len(COMMON[arm])} items answered by all {len(MODEL_IDS)} models')

# ---- 4. metrics ------------------------------------------------------------
def metrics(mid, arm, subset=None):
    d = load(mid, arm)
    ids = [c for c in COMMON[arm] if subset is None or d[c][1] in subset]
    if not ids: return None
    y_true = [d[c][1] for c in ids]
    # unparseable -> a sentinel that can never match, so it scores as wrong
    y_pred = [d[c][0] if d[c][0] is not None else 'CWE-UNPARSED' for c in ids]
    labels = sorted(set(y_true))
    return {
        'n': len(ids),
        'micro': 100*np.mean([p == t for p, t in zip(y_pred, y_true)]),
        'macro_f1': 100*f1_score(y_true, y_pred, labels=labels,
                                 average='macro', zero_division=0),
        'balanced': 100*balanced_accuracy_score(y_true, y_pred),
        'macro_rec': 100*recall_score(y_true, y_pred, labels=labels,
                                      average='macro', zero_division=0),
    }

for arm in ('bench','post'):
    print(f'\n=== {arm.upper()} (all classes) ===')
    print(f'{"model":12s}{"n":>6s}{"micro":>9s}{"macroF1":>10s}{"balanced":>10s}{"macroRec":>10s}')
    for mid in MODEL_IDS:
        m = metrics(mid, arm)
        if m:
            print(f'{mid:12s}{m["n"]:6d}{m["micro"]:9.1f}{m["macro_f1"]:10.1f}'
                  f'{m["balanced"]:10.1f}{m["macro_rec"]:10.1f}')

# ---- 5. class support and the frequent/rare split --------------------------
ref = MODEL_IDS[0]
for arm in ('bench','post'):
    d = load(ref, arm)
    sup = Counter(d[c][1] for c in COMMON[arm])
    freq = {c for c, n in sup.items() if n >= MIN_SUPPORT}
    rare = set(sup) - freq
    nf = sum(sup[c] for c in freq); nr = sum(sup[c] for c in rare)
    print(f'\n=== {arm.upper()} support: {len(sup)} classes; '
          f'{len(freq)} with n>={MIN_SUPPORT} ({nf} items), '
          f'{len(rare)} rarer ({nr} items) ===')
    print(f'{"model":12s}{"micro freq":>12s}{"micro rare":>12s}{"macroF1 freq":>14s}')
    for mid in MODEL_IDS:
        a = metrics(mid, arm, subset=freq)
        b = metrics(mid, arm, subset=rare)
        if a and b:
            print(f'{mid:12s}{a["micro"]:12.1f}{b["micro"]:12.1f}{a["macro_f1"]:14.1f}')

# ---- 6. per-class accuracy for well-supported classes ----------------------
arm = 'bench'
d0 = load(ref, arm)
sup = Counter(d0[c][1] for c in COMMON[arm])
top = [c for c, _ in sup.most_common() if sup[c] >= MIN_SUPPORT]
print(f'\n=== PER-CLASS ACCURACY, {arm}, classes with n>={MIN_SUPPORT} ===')
hdr = f'{"CWE":12s}{"n":>5s}' + ''.join(f'{m[:8]:>10s}' for m in MODEL_IDS) + f'{"range":>8s}'
print(hdr)
spreads = []
for cwe in top:
    row, accs = f'{cwe:12s}{sup[cwe]:5d}', []
    for mid in MODEL_IDS:
        d = load(mid, arm)
        ids = [c for c in COMMON[arm] if d[c][1] == cwe]
        a = 100*np.mean([d[c][0] == cwe for c in ids])
        accs.append(a); row += f'{a:10.1f}'
    sp = max(accs) - min(accs); spreads.append((cwe, sup[cwe], sp))
    print(row + f'{sp:8.1f}')

print(f'\nper-class range across models: median {np.median([s for _,_,s in spreads]):.1f} pp, '
      f'max {max(s for _,_,s in spreads):.1f} pp')
worst = sorted(spreads, key=lambda x: -x[2])[:3]
print('largest per-class disagreement:')
for cwe, n, sp in worst:
    print(f'   {cwe} (n={n}): {sp:.1f} pp')
print('\nIf per-class ranges are much wider than the 1.5 pp aggregate range,')
print('aggregate equivalence conceals per-class differences -- report both.')

# ---- 7. corrected null for duplicate label concordance ---------------------
sup_b = Counter(load(ref,'bench')[c][1] for c in COMMON['bench'])
tot = sum(sup_b.values())
p_same = sum((n/tot)**2 for n in sup_b.values())
print(f'\n=== corrected null for near-duplicate concordance ===')
print(f'majority-class rate            : {100*max(sup_b.values())/tot:.1f}%')
print(f'P(same label | random pair)    : {100*p_same:.1f}%   <- correct comparator')
print('Compare the observed 85.0% duplicate-pair concordance against this,')
print('not against the majority-class rate.')

# ---- 8. parser diagnostics -------------------------------------------------
print(f'\n=== parser diagnostics ===')
print(f'{"model":12s}{"arm":7s}{"strict":>9s}{"fallback":>10s}{"unparsed":>10s}{"multiCWE":>10s}')
for mid in MODEL_IDS:
    for arm in ('bench','post'):
        p = _RES / f'{mid}__rcm__{arm}.jsonl'
        if not p.exists(): continue
        want = set(ARM_IDS[arm])
        strict = fb = unp = multi = 0
        for line in open(p, encoding='utf-8'):
            if not line.strip(): continue
            r = json.loads(line)
            if r['cve'] not in want or not r.get('response'): continue
            t = r['response']
            hits = _CWE.findall(t)
            if len(set(hits)) > 1: multi += 1
            ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
            if ls and _CWE.fullmatch(ls[-1].strip(' .*`')): strict += 1
            elif hits: fb += 1
            else: unp += 1
        print(f'{mid:12s}{arm:7s}{strict:9d}{fb:10d}{unp:10d}{multi:10d}')
print('\n"fallback" = scored via the last-CWE-mention rule rather than the')
print('final-line rule. A high fallback rate for one model is a confound.')

# NOTE: the TF-IDF baseline is not included here -- its per-item predictions
# were never written to disk. To add its macro metrics, re-run Stage 1b with a
# line that saves clf.predict(...) alongside the gold labels.

arms from arms_N999_2026-06-01.json: bench=997, post=997
models found: ['claude', 'gpt-4.1', 'gpt-4o', 'gpt-5.5', 'llama-4']
bench: 997 items answered by all 5 models
post: 996 items answered by all 5 models

=== BENCH (all classes) ===
model            n    micro   macroF1  balanced  macroRec
claude         997     74.3      52.2      51.9      51.9
gpt-4.1        997     73.8      48.6      50.3      50.3
gpt-4o         997     75.3      50.2      52.5      52.5
gpt-5.5        997     74.9      53.6      54.1      54.1
llama-4        997     74.6      51.2      52.0      52.0

=== POST (all classes) ===
model            n    micro   macroF1  balanced  macroRec
claude         996     76.5      54.6      56.7      56.7
gpt-4.1        996     74.5      54.2      56.9      56.9
gpt-4o         996     75.7      51.7      54.4      54.4
gpt-5.5        996     77.2      58.7      61.2      61.2
llama-4        996     74.5      49.7      51.8      51.8

=== BENCH support: 102 classes; 23 wit

In [4]:
import json, re, collections
from pathlib import Path
_RES = Path('/content/drive/MyDrive/ctibench-audit/results')
_CWE = re.compile(r'CWE-\d+')
def parse(t):
    if not t: return None
    ls=[l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m=_CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h=_CWE.findall(t); return h[-1] if h else None

want = set(ARM_IDS['bench'])
for mid in ['gpt-4o','gpt-5.5','claude']:
    preds = collections.Counter()
    for line in open(_RES/f'{mid}__rcm__bench.jsonl', encoding='utf-8'):
        if not line.strip(): continue
        r = json.loads(line)
        if r['cve'] in want and r.get('response') and r['gt'].strip()=='CWE-787':
            preds[parse(r['response'])] += 1
    print(f'{mid}: {preds.most_common(6)}')

gpt-4o: [('CWE-121', 48), ('CWE-122', 32), ('CWE-787', 28), ('CWE-119', 8), ('CWE-20', 7), ('CWE-125', 4)]
gpt-5.5: [('CWE-121', 45), ('CWE-787', 38), ('CWE-122', 25), ('CWE-20', 4), ('CWE-130', 3), ('CWE-124', 3)]
claude: [('CWE-121', 26), ('CWE-787', 15), ('CWE-122', 10), ('CWE-20', 5), ('CWE-120', 3), ('CWE-129', 1)]


In [5]:
# ============================================================================
# HIERARCHY-AWARE SCORING
#
# Downloads the official CWE catalogue, builds the ChildOf relation graph, and
# re-scores every model under four criteria:
#   exact         prediction == gold
#   parent/child  prediction is a direct parent or direct child of gold
#   subtree       prediction is any ancestor or any descendant of gold
#   family        prediction and gold share a common ancestor within DEPTH hops
#
# Self-contained: needs only results/*.jsonl and derived/arms_*.json on Drive.
# ============================================================================

DEPTH = 2        # hops for the "family" criterion
import json, re, io, zipfile, urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict, deque
import numpy as np

_PROJ = Path('/content/drive/MyDrive/ctibench-audit')
_RES, _DER = _PROJ / 'results', _PROJ / 'derived'
CACHE = _DER / 'cwec_relations.json'

# ---- 1. CWE hierarchy ------------------------------------------------------
if CACHE.exists():
    rel = json.loads(CACHE.read_text())
    PARENTS = {k: set(v) for k, v in rel['parents'].items()}
    print(f'CWE relations loaded from cache ({len(PARENTS)} entries)')
else:
    URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'
    print('downloading CWE catalogue...')
    raw = urllib.request.urlopen(URL, timeout=120).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        xml = z.read([n for n in z.namelist() if n.endswith('.xml')][0])
    root = ET.fromstring(xml)
    ns = {'c': root.tag.split('}')[0].strip('{')} if '}' in root.tag else {}
    def tag(e): return e.tag.split('}')[-1]
    PARENTS = defaultdict(set)
    seen = set()
    for w in root.iter():
        if tag(w) not in ('Weakness', 'Category'): continue
        wid = w.get('ID')
        if not wid: continue
        cid = f'CWE-{wid}'; seen.add(cid)
        for rel_el in w.iter():
            if tag(rel_el) != 'Related_Weakness': continue
            if rel_el.get('Nature') == 'ChildOf' and rel_el.get('CWE_ID'):
                PARENTS[cid].add(f"CWE-{rel_el.get('CWE_ID')}")
    PARENTS = {k: set(v) for k, v in PARENTS.items()}
    for c in seen:
        PARENTS.setdefault(c, set())
    _DER.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps({'parents': {k: sorted(v) for k, v in PARENTS.items()}}))
    print(f'parsed {len(PARENTS)} CWE entries, cached to {CACHE.name}')

CHILDREN = defaultdict(set)
for c, ps in PARENTS.items():
    for p in ps:
        CHILDREN[p].add(c)

def ancestors(c, limit=None):
    out, q, d = set(), deque([(c, 0)]), None
    while q:
        n, dep = q.popleft()
        if limit is not None and dep >= limit: continue
        for p in PARENTS.get(n, ()):
            if p not in out:
                out.add(p); q.append((p, dep + 1))
    return out

def descendants(c, limit=None):
    out, q = set(), deque([(c, 0)])
    while q:
        n, dep = q.popleft()
        if limit is not None and dep >= limit: continue
        for ch in CHILDREN.get(n, ()):
            if ch not in out:
                out.add(ch); q.append((ch, dep + 1))
    return out

def relation(pred, gold):
    if pred is None: return 'unparsed'
    if pred == gold: return 'exact'
    if pred in PARENTS.get(gold, set()): return 'child_of_pred'   # pred is parent of gold
    if pred in CHILDREN.get(gold, set()): return 'parent_of_pred' # pred is child of gold
    if pred in ancestors(gold) or pred in descendants(gold): return 'subtree'
    if ancestors(gold, DEPTH) & ancestors(pred, DEPTH): return 'family'
    return 'unrelated'

# ---- 2. load outcomes ------------------------------------------------------
arm_files = sorted(_DER.glob('arms_*.json'))
assert arm_files, 'no arms_*.json found'
_s = json.loads(arm_files[-1].read_text())
ARM_IDS = {'bench': list(_s['bench']), 'post': list(_s['post'])}
MODEL_IDS = sorted({p.name.split('__')[0] for p in _RES.glob('*__rcm__*.jsonl')})
print(f'models: {MODEL_IDS}')

_CWE = re.compile(r'CWE-\d+')
def parse(t):
    if not t: return None
    ls = [l.strip() for l in t.strip().split('\n') if l.strip()]
    if ls:
        m = _CWE.fullmatch(ls[-1].strip(' .*`'))
        if m: return m.group(0)
    h = _CWE.findall(t)
    return h[-1] if h else None

def load(mid, arm):
    """dict cve -> (pred, gold); LAST record per CVE, so re-runs don't double-count."""
    p = _RES / f'{mid}__rcm__{arm}.jsonl'
    want, out = set(ARM_IDS[arm]), {}
    if not p.exists(): return out
    for line in open(p, encoding='utf-8'):
        if not line.strip(): continue
        r = json.loads(line)
        if r['cve'] in want and r.get('response'):
            out[r['cve']] = (parse(r['response']), r['gt'].strip())
    return out

COMMON = {}
for arm in ('bench', 'post'):
    sets = [set(load(m, arm)) for m in MODEL_IDS]
    COMMON[arm] = sorted(set(ARM_IDS[arm]).intersection(*[s for s in sets if s]))
    print(f'{arm}: {len(COMMON[arm])} items')

# ---- 3. re-score -----------------------------------------------------------
def score(mid, arm):
    d = load(mid, arm)
    rels = [relation(*d[c]) for c in COMMON[arm]]
    n = len(rels); C = Counter(rels)
    exact = C['exact']
    pc    = exact + C['parent_of_pred'] + C['child_of_pred']
    sub   = pc + C['subtree']
    fam   = sub + C['family']
    return {'n': n, 'exact': 100*exact/n, 'pc': 100*pc/n,
            'subtree': 100*sub/n, 'family': 100*fam/n,
            'unrelated': 100*C['unrelated']/n, 'unparsed': 100*C['unparsed']/n}

for arm in ('bench', 'post'):
    print(f'\n=== {arm.upper()}: accuracy under hierarchy-aware criteria ===')
    print(f'{"model":12s}{"exact":>8s}{"+par/chi":>10s}{"+subtree":>10s}'
          f'{"+family":>9s}{"unrel":>8s}{"unpars":>8s}')
    rows = {}
    for mid in MODEL_IDS:
        s = score(mid, arm); rows[mid] = s
        print(f'{mid:12s}{s["exact"]:8.1f}{s["pc"]:10.1f}{s["subtree"]:10.1f}'
              f'{s["family"]:9.1f}{s["unrelated"]:8.1f}{s["unparsed"]:8.1f}')
    for k, lbl in (('exact','exact'), ('pc','parent/child'), ('subtree','subtree')):
        v = [rows[m][k] for m in MODEL_IDS]
        print(f'  range across models, {lbl:13s}: {max(v)-min(v):.1f} pp')
    lift = np.mean([rows[m]['pc'] - rows[m]['exact'] for m in MODEL_IDS])
    print(f'  mean lift from allowing parent/child: {lift:+.1f} pp')

# ---- 4. which classes drive the lift ---------------------------------------
arm = 'bench'
print(f'\n=== classes where parent/child credit changes the most ({arm}) ===')
ref = load(MODEL_IDS[0], arm)
sup = Counter(ref[c][1] for c in COMMON[arm])
gain = []
for cwe, n in sup.items():
    if n < 10: continue
    ids = [c for c in COMMON[arm] if ref[c][1] == cwe]
    e = pc = 0
    for mid in MODEL_IDS:
        d = load(mid, arm)
        for c in ids:
            r = relation(*d[c])
            if r == 'exact': e += 1; pc += 1
            elif r in ('parent_of_pred','child_of_pred'): pc += 1
    tot = len(ids)*len(MODEL_IDS)
    gain.append((cwe, n, 100*e/tot, 100*pc/tot, 100*(pc-e)/tot))
gain.sort(key=lambda x: -x[4])
print(f'{"CWE":12s}{"n":>5s}{"exact":>9s}{"+par/chi":>10s}{"lift":>8s}')
for cwe, n, e, pc, g in gain[:10]:
    print(f'{cwe:12s}{n:5d}{e:9.1f}{pc:10.1f}{g:8.1f}')

print('\nA large lift on a well-supported class means models are answering at a')
print('different granularity than the reference label, not failing to identify')
print('the weakness family. Report exact-match as primary and hierarchy-aware')
print('as a sensitivity analysis; do not silently switch to the looser metric.')

downloading CWE catalogue...
parsed 1391 CWE entries, cached to cwec_relations.json
models: ['claude', 'gpt-4.1', 'gpt-4o', 'gpt-5.5', 'llama-4']
bench: 997 items
post: 996 items

=== BENCH: accuracy under hierarchy-aware criteria ===
model          exact  +par/chi  +subtree  +family   unrel  unpars
claude          74.3      84.2      85.7     92.3     7.7     0.0
gpt-4.1         73.8      84.4      85.5     92.5     7.5     0.0
gpt-4o          75.3      84.7      86.0     92.8     7.2     0.0
gpt-5.5         74.9      84.8      86.0     92.5     7.5     0.0
llama-4         74.6      85.1      86.5     93.0     6.9     0.1
  range across models, exact        : 1.5 pp
  range across models, parent/child : 0.9 pp
  range across models, subtree      : 1.0 pp
  mean lift from allowing parent/child: +10.0 pp

=== POST: accuracy under hierarchy-aware criteria ===
model          exact  +par/chi  +subtree  +family   unrel  unpars
claude          76.5      82.4      83.3     90.8     9.2     0.